In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

### MONTH 1 ANALYSIS

In [2]:
### MONTH 1 

m1Revenue_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='M1 Revenue')
m1Activities_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='M1 Activities')
m1Labour_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='M1 Labour')
m1Exceptions_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='M1 Exceptions')
m1Inventory_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='M1 Inventory')

In [3]:
# warehouse customer, pricing , product data

customer_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='Customer Master')
pricing_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='Standard Pricing')
customer_pricing_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='Customer Pricing')

cost_allocation_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='Cost Allocation')
management_allocation_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='Management Allocation Cost')
product_master_df=pd.read_excel('../Resources/warehouse_dataset.xlsx',sheet_name='Product Master')

### KNOW YOUR : CUSTOMERS

In [4]:
customer_df

,Customer ID,Customer Name,Industry,Contract Type,Internal Archetype
0,C001,Alpha Retail,Retail,Activity Based,Strategic Account
1,C002,Bravo FMCG,FMCG,Activity Based,Scale Without Return
2,C003,Charlie Medical,Healthcare,Activity Based,Revenue Leakage
3,C004,Delta Manufacturing,Industrial,Hybrid,Operational Complexity
4,C005,Echo Imports,Consumer Goods,Storage Heavy,Stable Performer
5,C006,Medisupply Australia,Healthcare,Hybrid,Stable Performer
6,C007,Precision Industrial Group,Industrial,Hybrid,Stable Performer
7,C008,National Wholesale Distributor,Wholesale,Activity Based,Stable Performer
8,C009,BrightHome Products,Consumer Goods,Activity Based,Stable Performer
9,C010,Southern Manufacturing Services,Industrial,Fixed Monthly,Stable Performer


In [5]:
customer_df.shape

(30, 5)

In [6]:
customer_df.dtypes

Customer ID           str
Customer Name         str
Industry              str
Contract Type         str
Internal Archetype    str
dtype: object

In [7]:
customer_df=customer_df.astype('object')
customer_df.dtypes

Customer ID           object
Customer Name         object
Industry              object
Contract Type         object
Internal Archetype    object
dtype: object

In [8]:
for column in customer_df.columns[2:]:
    print(f"{column} has {customer_df[column].unique()}\n")

Industry has ['Retail' 'FMCG' 'Healthcare' 'Industrial' 'Consumer Goods' 'Wholesale']

Contract Type has ['Activity Based' 'Hybrid' 'Storage Heavy' 'Fixed Monthly']

Internal Archetype has ['Strategic Account' 'Scale Without Return' 'Revenue Leakage'
 'Operational Complexity' 'Stable Performer' 'Storage Intensive'
 'Space Inefficient' 'Fixed Fee Risk' 'Seasonal Volatility'
 'Activity Intensive' 'Margin Expansion Opportunity'
 'Underutilised Account' 'Service Intensive']



In [ ]:
import pandas as pd
import plotly.express as px

# ==========================================
# 1. CORE COLOR MAPPING & CONFIGURATION
# ==========================================
# Centralized color map to keep visualizations consistent for executive presentation
archetype_colors = {
    # The Protect Zone (Greens / Teal)
    "Stable Performer": "#2ecc71",               # Professional Green
    "Strategic Account": "#27ae60",              # Premium Dark Green
    "Margin Expansion Opportunity": "#a3e4d7",   # Light Teal/Mint
    
    # The Renegotiate Zone (Yellows / Oranges / Purples)
    "Scale Without Return": "#f39c12",           # Soft Amber
    "Activity Intensive": "#e67e22",             # Deep Orange
    "Operational Complexity": "#d35400",         # Dark Burnt Orange
    "Service Intensive": "#884ea0",              # Dark Magenta/Burgundy
    
    # The Optimize Zone (Blues / Grays)
    "Underutilised Account": "#3498db",          # Corporate Blue
    "Storage Intensive": "#2980b9",              # Dark Blue
    "Space Inefficient": "#5d6d7e",              # Muted Slate Gray
    
    # The Danger Zone (Reds / Purples)
    "Revenue Leakage": "#e74c3c",                # Alert Red
    "Fixed Fee Risk": "#c0392b",                 # Crimson/High Risk
    "Seasonal Volatility": "#9b59b6"             # Purple
}

# ==========================================
# 2. VISUALIZATION 1: INDUSTRY PIE CHART
# ==========================================
# Pre-aggregate data for clean layout rendering
industry_counts = customer_df['Industry'].value_counts().reset_index(name='Count')

fig_pie = px.pie(
    industry_counts, 
    values='Count', 
    names='Industry', 
    title='<b>Customer Portfolio Mix by Industry</b>',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_pie.update_traces(textinfo='percent+label')
fig_pie.update_layout(margin=dict(t=60, b=30, l=20, r=20))
fig_pie.show()


# ==========================================
# 3. VISUALIZATION 2: CONTRACT TYPE COUNTPLOT
# ==========================================
# Pre-aggregate to lock descending sort order automatically
contract_counts = customer_df['Contract Type'].value_counts().reset_index(name='Count')

fig_bar = px.bar(
    contract_counts, 
    x='Contract Type', 
    y='Count', 
    title='<b>Analysis of Active Contract Types</b>',
    color='Contract Type',
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_bar.update_layout(
    xaxis_title="Contract Type",
    yaxis_title="Number of Customers",
    showlegend=False,
    margin=dict(t=60, b=30, l=20, r=20)
)
fig_bar.show()


# ==========================================
# 4. VISUALIZATION 3: ARCHETYPE BUBBLE CHART
# ==========================================
# Pre-aggregate to size the bubbles properly
archetype_counts = customer_df['Internal Archetype'].value_counts().reset_index(name='Count')

fig_bubble = px.scatter(
    archetype_counts, 
    x='Internal Archetype', 
    y='Count',
    size='Count', 
    color='Internal Archetype',
    color_discrete_map=archetype_colors,  # Explicitly applying the matrix colors
    title='<b>Management Archetype Concentration (Risk & Opportunity Flags)</b>',
    size_max=50
)
fig_bubble.update_layout(
    xaxis_title="Internal Archetype",
    yaxis_title="Customer Count",
    showlegend=False,
    height=550,   
    margin=dict(t=60, b=120, l=20, r=20) 
)
fig_bubble.update_xaxes(tickangle=45)
fig_bubble.show()


import plotly.express as px
import pandas as pd

# ==========================================
# Improved Treemap with Better Colors
# ==========================================

treemap_data = (
    customer_df.groupby(['Industry', 'Contract Type', 'Internal Archetype'])
    .size()
    .reset_index(name='Count')
)

total = treemap_data['Count'].sum()
treemap_data['Percentage'] = (treemap_data['Count'] / total * 100).round(1)

fig_tree = px.treemap(
    treemap_data,
    path=['Industry', 'Contract Type', 'Internal Archetype'],
    values='Count',
    color='Industry',                           # ← Better: Color by Industry (cleaner & more professional)
    color_discrete_sequence=px.colors.qualitative.Bold,   # Stronger, modern colors
    title="<b>Customer Portfolio Structural Breakdown</b><br><sup>By Industry → Contract Type → Internal Archetype</sup>",
)

fig_tree.update_traces(
    textinfo="label+value+percent parent",
    textfont=dict(size=12, family="Arial", color="white"),
    hovertemplate="<b>%{label}</b><br>Count: %{value}<br>Share: %{percentParent:.1%}<br><extra></extra>",
    marker=dict(line=dict(width=2, color="white"))
)

fig_tree.update_layout(
    height=650,
    margin=dict(t=70, l=20, r=20, b=20),
    font=dict(family="Arial", size=11),
    title_font=dict(size=18, family="Arial", color="#1a1a2e"),
)

fig_tree.show()

In [10]:
import plotly.graph_objects as go
import pandas as pd

# ============================================
# DATA: 13 Archetypes positioned on the matrix
# ============================================
archetypes = [
    # Profit Stars (High Revenue, Low Cost)
    {"archetype": "Strategic Account", "revenue": 8.5, "cost": 2.0, "quadrant": "Profit Star", 
     "desc": "High-value, efficient, long-term client. Protect & grow."},
    {"archetype": "Stable Performer", "revenue": 7.2, "cost": 3.0, "quadrant": "Profit Star", 
     "desc": "Consistent revenue with balanced, low-drama operations."},
    {"archetype": "Margin Expansion Opportunity", "revenue": 7.8, "cost": 4.0, "quadrant": "Profit Star", 
     "desc": "Good base with clear upside via pricing or efficiency."},

    # High Maintenance (High Revenue, High Cost)
    {"archetype": "Operational Complexity", "revenue": 8.0, "cost": 7.5, "quadrant": "High Maintenance", 
     "desc": "High revenue but disproportionate effort and exceptions."},
    {"archetype": "Service Intensive", "revenue": 7.3, "cost": 8.3, "quadrant": "High Maintenance", 
     "desc": "Labor-intensive services need proper costing."},
    {"archetype": "Activity Intensive", "revenue": 8.7, "cost": 7.2, "quadrant": "High Maintenance", 
     "desc": "High throughput. Labor cost recovery is critical."},
    {"archetype": "Storage Intensive", "revenue": 7.0, "cost": 7.8, "quadrant": "High Maintenance", 
     "desc": "High storage revenue but heavy space & holding cost."},
    {"archetype": "Seasonal Volatility", "revenue": 7.9, "cost": 8.5, "quadrant": "High Maintenance", 
     "desc": "Peak revenue but expensive capacity strain & idle periods."},
    {"archetype": "Fixed Fee Risk", "revenue": 8.3, "cost": 7.3, "quadrant": "High Maintenance", 
     "desc": "Large accounts on fixed pricing exposed to cost overruns."},

    # Value Destroyers (Low Revenue, High Cost)
    {"archetype": "Revenue Leakage", "revenue": 3.8, "cost": 8.2, "quadrant": "Value Destroyer", 
     "desc": "Costs exceed revenue. Direct profit destruction."},
    {"archetype": "Scale Without Return", "revenue": 5.8, "cost": 7.6, "quadrant": "Value Destroyer", 
     "desc": "Growing revenue but costs growing faster. Unprofitable growth."},
    {"archetype": "Space Inefficient", "revenue": 4.2, "cost": 7.1, "quadrant": "Value Destroyer", 
     "desc": "Consumes disproportionate space relative to revenue."},

    # Low Impact (Low Revenue, Low Cost)
    {"archetype": "Underutilised Account", "revenue": 3.2, "cost": 3.5, "quadrant": "Low Impact", 
     "desc": "Low activity but occupying space/resources. Opportunity cost."},
]

df = pd.DataFrame(archetypes)

# Color mapping by quadrant
color_map = {
    "Profit Star": "#2E7D32",       # Green
    "High Maintenance": "#F57C00",  # Orange
    "Value Destroyer": "#C62828",   # Red
    "Low Impact": "#5C6BC0"         # Indigo
}

df["color"] = df["quadrant"].map(color_map)

# ============================================
# CREATE INTERACTIVE PLOTLY MATRIX
# ============================================
fig = go.Figure()

# Add quadrant background rectangles
fig.add_shape(type="rect", x0=0, y0=5, x1=5, y1=10, fillcolor="#FFEBEE", opacity=0.25, line_width=0)  # Value Destroyer
fig.add_shape(type="rect", x0=5, y0=5, x1=10, y1=10, fillcolor="#FFF3E0", opacity=0.25, line_width=0) # High Maintenance
fig.add_shape(type="rect", x0=0, y0=0, x1=5, y1=5, fillcolor="#E8EAF6", opacity=0.25, line_width=0)  # Low Impact
fig.add_shape(type="rect", x0=5, y0=0, x1=10, y1=5, fillcolor="#E8F5E9", opacity=0.25, line_width=0) # Profit Star

# Add scatter points
fig.add_trace(go.Scatter(
    x=df["revenue"],
    y=df["cost"],
    mode="markers+text",
    text=df["archetype"],
    textposition="top center",
    textfont=dict(size=10, color="black"),
    marker=dict(
        size=14,
        color=df["color"],
        line=dict(width=1.5, color="white"),
        symbol="circle"
    ),
    customdata=df[["quadrant", "desc"]],
    hovertemplate=
        "<b>%{text}</b><br>" +
        "Quadrant: %{customdata[0]}<br>" +
        "Revenue: %{x:.1f} | Cost to Serve: %{y:.1f}<br>" +
        "%{customdata[1]}<br>" +
        "<extra></extra>",
    name="Archetypes"
))

# Add quadrant labels as annotations
fig.add_annotation(x=2.5, y=9.2, text="<b>Value Destroyers</b><br>(Low Revenue, High Cost)", 
                   showarrow=False, font=dict(size=13, color="#B71C1C"), align="center")
fig.add_annotation(x=7.5, y=9.2, text="<b>High Maintenance</b><br>(High Revenue, High Cost)", 
                   showarrow=False, font=dict(size=13, color="#E65100"), align="center")
fig.add_annotation(x=2.5, y=1.0, text="<b>Low Impact / Filler</b><br>(Low Revenue, Low Cost)", 
                   showarrow=False, font=dict(size=13, color="#283593"), align="center")
fig.add_annotation(x=7.5, y=1.0, text="<b>Profit Stars</b><br>(High Revenue, Low Cost)", 
                   showarrow=False, font=dict(size=13, color="#1B5E20"), align="center")

# Layout settings
fig.update_layout(
    title=dict(
        text="<b>Warehouse Account Archetype Impact Matrix</b><br>Revenue vs Cost to Serve",
        font=dict(size=18),
        x=0.5
    ),
    xaxis=dict(
        title="Revenue Contribution",
        range=[0, 10],
        dtick=2,
        gridcolor="lightgray",
        zerolinecolor="gray"
    ),
    yaxis=dict(
        title="Cost to Serve",
        range=[0, 10],
        dtick=2,
        gridcolor="lightgray",
        zerolinecolor="gray"
    ),
    width=950,
    height=750,
    plot_bgcolor="white",
    paper_bgcolor="white",
    hoverlabel=dict(bgcolor="white", font_size=12),
    legend=dict(
        title="Quadrant",
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    ),
    margin=dict(l=60, r=40, t=80, b=80)
)

# Add quadrant legend items (invisible traces for legend)
for quad, color in color_map.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=12, color=color),
        name=quad,
        showlegend=True
    ))

fig.show()

### CUSTOMERS: PRICING 

In [11]:
customer_pricing_df

,Customer Id,Activity Type,Unit Type,Contract Rate
0,C001,Storage,pallet_day,8.96
1,C001,Receipt,pallet,3.92
2,C001,Dispatch,order,4.48
3,C001,Pick,pick,0.17
4,C001,Returns,return,11.20
...,...,...,...,...
205,C030,Dispatch,order,4.43
206,C030,Pick,pick,0.14
207,C030,Returns,return,8.54
208,C030,Rework,job,11.43


In [12]:
# Quick one-liner checks
print("Shape:", customer_pricing_df.shape)
print("\nColumns:", customer_pricing_df.columns.tolist())
print("\nDtypes:\n", customer_pricing_df.dtypes)
print("\nMissing Values:\n", customer_pricing_df.isnull().sum())
print("\nDuplicate Rows:", customer_pricing_df.duplicated().sum())

Shape: (210, 4)

Columns: ['Customer Id', 'Activity Type', 'Unit Type', 'Contract Rate']

Dtypes:
 Customer Id          str
Activity Type        str
Unit Type            str
Contract Rate    float64
dtype: object

Missing Values:
 Customer Id      0
Activity Type    0
Unit Type        0
Contract Rate    1
dtype: int64

Duplicate Rows: 0


In [13]:
customer_pricing_df[['Customer Id','Activity Type','Unit Type']]=customer_pricing_df[['Customer Id','Activity Type','Unit Type']].astype('object')
customer_pricing_df.dtypes

Customer Id       object
Activity Type     object
Unit Type         object
Contract Rate    float64
dtype: object

In [14]:
customer_pricing_df[customer_pricing_df['Contract Rate'].isnull()]

,Customer Id,Activity Type,Unit Type,Contract Rate
96,C014,Rework,job,NaN


In [15]:
activity_unit_df=customer_pricing_df[['Activity Type','Unit Type']].drop_duplicates().copy()
activity_unit_df


,Activity Type,Unit Type
0,Storage,pallet_day
1,Receipt,pallet
2,Dispatch,order
3,Pick,pick
4,Returns,return
5,Rework,job
6,Urgent Order,order


In [16]:
# Check the specific missing row
missing_row = customer_pricing_df[customer_pricing_df['Contract Rate'].isnull()]
display(missing_row)

# Check what other customers are charged for Rework
rework_rates = customer_pricing_df[customer_pricing_df['Activity Type'] == 'Rework']
print("\nRework rates across all customers:")
print(rework_rates[['Customer Id', 'Contract Rate']].sort_values('Contract Rate'))

# Check what rates C014 has for other activities
c014_rates = customer_pricing_df[customer_pricing_df['Customer Id'] == 'C014']
print("\nAll rates for C014:")
display(c014_rates)

,Customer Id,Activity Type,Unit Type,Contract Rate
96,C014,Rework,job,NaN



Rework rates across all customers:
    Customer Id  Contract Rate
12         C002           9.36
26         C004          10.20
194        C028          10.28
40         C006          10.29
145        C021          10.61
33         C005          10.68
68         C010          10.92
166        C024          10.94
138        C020          10.94
187        C027          10.97
173        C025          11.00
201        C029          11.26
152        C022          11.29
208        C030          11.43
117        C017          11.54
75         C011          11.58
110        C016          11.79
131        C019          11.82
19         C003          12.60
47         C007          12.67
61         C009          12.97
159        C023          12.98
54         C008          13.08
103        C015          13.32
124        C018          13.37
180        C026          13.40
82         C012          13.42
5          C001          13.44
89         C013          13.50
96         C014            NaN

Al

,Customer Id,Activity Type,Unit Type,Contract Rate
91,C014,Storage,pallet_day,8.58
92,C014,Receipt,pallet,3.02
93,C014,Dispatch,order,3.79
94,C014,Pick,pick,0.14
95,C014,Returns,return,9.67
96,C014,Rework,job,NaN
97,C014,Urgent Order,order,13.57


##### filling missing value

In [17]:

median_rework = customer_pricing_df[customer_pricing_df['Activity Type'] == 'Rework']['Contract Rate'].median()

print(f"Median Rework Rate across all customers: {median_rework:.2f}")
customer_pricing_df['Contract Rate'] = customer_pricing_df['Contract Rate'].fillna(median_rework)
print("\nMissing values after imputation:")
print(customer_pricing_df.isnull().sum())

print("\nC014 Rework row after filling:")
display(customer_pricing_df[(customer_pricing_df['Customer Id'] == 'C014') & 
                            (customer_pricing_df['Activity Type'] == 'Rework')])

Median Rework Rate across all customers: 11.54

Missing values after imputation:
Customer Id      0
Activity Type    0
Unit Type        0
Contract Rate    0
dtype: int64

C014 Rework row after filling:


,Customer Id,Activity Type,Unit Type,Contract Rate
96,C014,Rework,job,11.54


In [18]:
customer_pricing_df.isnull().sum()

Customer Id      0
Activity Type    0
Unit Type        0
Contract Rate    0
dtype: int64

In [19]:
import plotly.express as px

fig = px.box(
    customer_pricing_df,
    x='Activity Type',
    y='Contract Rate',
    color='Activity Type',
    title="<b>Contract Rate Distribution by Activity Type</b><br><sup>Shows pricing consistency across customers</sup>",
    points="all",                    # Show all individual customer rates
    hover_data=['Customer Id']
)

fig.update_layout(
    height=600,
    width=950,
    showlegend=False,
    yaxis_title="Contract Rate ($)",
    xaxis_title="Activity Type",
    title_font=dict(size=18),
    boxmode='group'
)

fig.show()

In [20]:
fig = px.strip(
    customer_pricing_df,
    x='Activity Type',
    y='Contract Rate',
    color='Activity Type',
    title="<b>Individual Customer Contract Rates by Activity</b>",
    hover_data=['Customer Id'],
    stripmode='overlay'
)

fig.update_layout(height=600, showlegend=False)
fig.show()

In [21]:
import plotly.express as px

avg_rate = customer_pricing_df.groupby('Activity Type')['Contract Rate'].mean().reset_index()

fig2 = px.bar(
    avg_rate,
    x='Activity Type',
    y='Contract Rate',
    color='Activity Type',
    title="<b>Average Contract Rate by Activity Type</b>",
    text=avg_rate['Contract Rate'].round(2)
)

fig2.update_traces(textposition='outside')
fig2.update_layout(
    height=550,
    showlegend=False,
    yaxis_title="Average Contract Rate ($)",
    title_font=dict(size=18)
)

fig2.show()

In [22]:


# ==========================================
# Enhanced Pricing Variation Summary (with CV)
# ==========================================

summary = customer_pricing_df.groupby('Activity Type').agg(
    No_of_Customers=('Customer Id', 'nunique'),
    Min_Rate=('Contract Rate', 'min'),
    Max_Rate=('Contract Rate', 'max'),
    Median_Rate=('Contract Rate', 'median'),
    Mean_Rate=('Contract Rate', 'mean'),
    Std_Dev=('Contract Rate', 'std')
).reset_index()

# Calculate both types of variation
summary['Range'] = summary['Max_Rate'] - summary['Min_Rate']
summary['Variation_MaxMin_%'] = ((summary['Max_Rate'] - summary['Min_Rate']) / summary['Median_Rate'] * 100).round(1)
summary['CV_%'] = ((summary['Std_Dev'] / summary['Mean_Rate']) * 100).round(1)

# Clean column names
summary = summary.rename(columns={
    'Activity Type': 'Activity',
    'Min_Rate': 'Min ($)',
    'Max_Rate': 'Max ($)',
    'Median_Rate': 'Median ($)',
    'Mean_Rate': 'Mean ($)',
    'Std_Dev': 'Std Dev'
})

# Sort by CV (more statistical)
summary = summary.sort_values('CV_%', ascending=False).reset_index(drop=True)

# Round values
summary = summary.round(2)

print("=== PRICING VARIATION SUMMARY ===\n")
display(summary[['Activity', 'No_of_Customers', 'Min ($)', 'Max ($)', 'Median ($)', 'Variation_MaxMin_%', 'CV_%']])

=== PRICING VARIATION SUMMARY ===



,Activity,No_of_Customers,Min ($),Max ($),Median ($),Variation_MaxMin_%,CV_%
0,Storage,30,6.24,10.40,8.12,51.2,10.9
1,Urgent Order,30,11.70,17.22,15.12,36.5,10.3
2,Rework,30,9.36,13.50,11.54,35.9,10.2
3,Pick,30,0.12,0.17,0.15,33.3,10.0
4,Receipt,30,2.73,4.02,3.36,38.3,10.0
5,Returns,30,7.80,11.48,9.73,37.8,9.4
6,Dispatch,30,3.12,4.57,3.93,36.9,9.0


In [23]:
summary.columns

Index(['Activity', 'No_of_Customers', 'Min ($)', 'Max ($)', 'Median ($)',
       'Mean ($)', 'Std Dev', 'Range', 'Variation_MaxMin_%', 'CV_%'],
      dtype='str')

In [24]:
summary[['Activity','Variation_MaxMin_%','CV_%']]

,Activity,Variation_MaxMin_%,CV_%
0,Storage,51.2,10.9
1,Urgent Order,36.5,10.3
2,Rework,35.9,10.2
3,Pick,33.3,10.0
4,Receipt,38.3,10.0
5,Returns,37.8,9.4
6,Dispatch,36.9,9.0


In [25]:
customer_pricing_df

,Customer Id,Activity Type,Unit Type,Contract Rate
0,C001,Storage,pallet_day,8.96
1,C001,Receipt,pallet,3.92
2,C001,Dispatch,order,4.48
3,C001,Pick,pick,0.17
4,C001,Returns,return,11.20
...,...,...,...,...
205,C030,Dispatch,order,4.43
206,C030,Pick,pick,0.14
207,C030,Returns,return,8.54
208,C030,Rework,job,11.43


In [26]:
import plotly.express as px

# Pivot the data for heatmap
heatmap_data = customer_pricing_df.pivot(
    index='Customer Id', 
    columns='Activity Type', 
    values='Contract Rate'
)

# Create the heatmap
fig = px.imshow(
    heatmap_data,
    color_continuous_scale='Viridis',        # Good options: 'Viridis', 'Plasma', 'Blues', 'RdYlGn'
    aspect="auto",
    title="<b>Contract Rates Heatmap</b><br><sup>All Customers (C001–C030) × All Activities</sup>",
    labels=dict(
        x="Activity Type", 
        y="Customer", 
        color="Contract Rate ($)"
    )
)

# Improve layout
fig.update_layout(
    height=750,
    width=950,
    title_font=dict(size=18),
    xaxis_title="Activity Type",
    yaxis_title="Customer ID",
    coloraxis_colorbar=dict(
        title="Rate ($)",
        thickness=15,
        len=0.8
    )
)

# Add hover information
fig.update_traces(
    hovertemplate="<b>Customer:</b> %{y}<br><b>Activity:</b> %{x}<br><b>Rate:</b> $%{z:.2f}<extra></extra>"
)

fig.show()

### COST ALLOCATION :WHAT'S KEEPING WAREHOUSE DOOR OPEN

In [27]:
cost_allocation_df_m1=cost_allocation_df[cost_allocation_df['Month']=='Month 1']
cost_allocation_df_m1

,Month,Cost Category,Monthly Cost,Cost Type
0,Month 1,Rent,52000,Fixed
1,Month 1,Utilities,12500,Semi-variable
2,Month 1,Equipment,18500,Fixed
3,Month 1,Warehouse Management,34000,Fixed
4,Month 1,Corporate Overhead,28000,Allocated


In [28]:
import pandas as pd
import plotly.graph_objects as go

# Step 1: Re-create your exact cost allocation data frame
cost_data = {
    'Month': ['Month 1', 'Month 1', 'Month 1', 'Month 1', 'Month 1'],
    'Cost Category': ['Rent', 'Utilities', 'Equipment', 'Warehouse Management', 'Corporate Overhead'],
    'Monthly Cost': [52000, 12500, 18500, 34000, 28000],
    'Cost Type': ['Fixed', 'Semi-variable', 'Fixed', 'Fixed', 'Allocated']
}
cost_allocation_m1 = pd.DataFrame(cost_data)
total_cost = cost_allocation_m1['Monthly Cost'].sum()
# Step 2: Pivot data into a clean structural matrix & fill empty blocks with 0
cost_matrix = cost_allocation_m1.pivot(
    index='Cost Category', 
    columns='Cost Type', 
    values='Monthly Cost'
).fillna(0)

# Step 3: Chronologically order columns from most rigid to most flexible
desired_order = ['Fixed', 'Semi-variable', 'Allocated']
cost_matrix = cost_matrix[[col for col in desired_order if col in cost_matrix.columns]]

# Step 4: Generate a matrix heatmap with explicit cost values
fig_matrix = go.Figure(data=go.Heatmap(
    z=cost_matrix.values,
    x=cost_matrix.columns,
    y=cost_matrix.index,
    colorscale='YlOrRd',  # Warm corporate gradient
    text=[[f"${val:,.0f}" if val > 0 else "-" for val in row] for row in cost_matrix.values],
    texttemplate="<b>%{text}</b>",  # Forces bold numbers inside cells
    textfont=dict(size=14),
    showscale=False,  # Hides unnecessary colorbar to keep it clean
    hoverongaps=False
))

# Step 5: Executive Layout Formats
fig_matrix.update_layout(
    title=dict(
        text="<b>Cost Allocation Matrix: Baseline Operational Commitments</b><br><sup>Categorizing Month 1 Facility-Sustaining Expenses</sup>",
        font=dict(size=20)
    ),
    xaxis_title="<b>Cost Classification (Financial Behavior) ➔</b>",
    yaxis_title="<b>Operating Cost Category ➔</b>",
    height=500,
    margin=dict(t=80, b=50, l=180, r=40) # Generous left margin to fit long names
)

fig_matrix.add_annotation(
    xref="paper", yref="paper",
    x=1.0, y=1.12,  # Places it beautifully in the upper-right area
    text=f"<b>TOTAL MONTH 1 OVERHEAD</b><br><span style='font-size:18px; color:#c0392b;'><b>${total_cost:,.0f}</b></span>",
    showarrow=False,
    align="center",
    bordercolor="#bdc3c7",
    borderwidth=1,
    borderpad=6,
    bgcolor="#fdfefe"
)

fig_matrix.show()

In [29]:
management_allocation_df_m1=management_allocation_df[management_allocation_df['Month']=='Month 1'].copy()
management_allocation_df_m1

,Month,Customer Id,Allocation Method,Management Allocated Cost
0,Month 1,C001,Revenue percentage,7442.366050
1,Month 1,C002,Revenue percentage,6311.317289
2,Month 1,C003,Revenue percentage,3530.021385
3,Month 1,C004,Revenue percentage,5744.316145
4,Month 1,C005,Revenue percentage,16464.452593
5,Month 1,C006,Revenue percentage,3384.693677
6,Month 1,C007,Revenue percentage,3183.881424
7,Month 1,C008,Revenue percentage,3674.910167
8,Month 1,C009,Revenue percentage,3915.559982
9,Month 1,C010,Revenue percentage,3190.000297


In [30]:
# ==========================================
# Management Allocation Analysis - Month 1
# ==========================================

# 1. Calculate Total Management Allocated Cost
total_allocated_cost = management_allocation_df_m1['Management Allocated Cost'].sum()

print(f"Total Management Allocated Cost (Month 1): ${total_allocated_cost:,.2f}")

# 2. Calculate % of total cost for each customer
management_allocation_df_m1['Overall_Cost_Percentage'] = (
    management_allocation_df_m1['Management Allocated Cost'] / total_allocated_cost * 100
).round(2)

# 3. Create a clean summary view
allocation_summary = management_allocation_df_m1[[
    'Customer Id', 
    'Management Allocated Cost', 
    'Overall_Cost_Percentage'
]].sort_values('Overall_Cost_Percentage', ascending=False).reset_index(drop=True)

# 4. Display results
print("\n=== Management Cost Allocation by Customer (Month 1) ===\n")
display(allocation_summary)

# Optional: Show top 5 customers by allocated cost %
print("\nTop 5 Customers by Management Cost Allocation %:")
display(allocation_summary.head(5))

Total Management Allocated Cost (Month 1): $132,500.00

=== Management Cost Allocation by Customer (Month 1) ===



,Customer Id,Management Allocated Cost,Overall_Cost_Percentage
0,C005,16464.452593,12.43
1,C001,7442.366050,5.62
2,C002,6311.317289,4.76
3,C004,5744.316145,4.34
4,C012,4476.272202,3.38
5,C013,4322.187583,3.26
6,C030,4275.662072,3.23
7,C016,4229.090691,3.19
8,C022,4107.564608,3.10
9,C014,3982.230264,3.01



Top 5 Customers by Management Cost Allocation %:


,Customer Id,Management Allocated Cost,Overall_Cost_Percentage
0,C005,16464.452593,12.43
1,C001,7442.366050,5.62
2,C002,6311.317289,4.76
3,C004,5744.316145,4.34
4,C012,4476.272202,3.38


In [31]:
import plotly.express as px
import pandas as pd

# ==========================================
# Horizontal Bar Chart - All 30 Customers
# ==========================================

# Prepare and sort data
bar_data = management_allocation_df_m1[['Customer Id', 'Management Allocated Cost']].copy()
bar_data = bar_data.sort_values('Management Allocated Cost', ascending=True)  # ascending for horizontal chart

# Calculate total for title
total_mgmt_cost = bar_data['Management Allocated Cost'].sum()

# Create Horizontal Bar Chart
fig = px.bar(
    bar_data,
    x='Management Allocated Cost',
    y='Customer Id',
    orientation='h',
    title=f"<b>Management Cost Allocation by Customer (Month 1)</b><br><sup>Total Allocated Cost: ${total_mgmt_cost:,.2f}</sup>",
    text=bar_data['Management Allocated Cost'].round(0),   # Show values on bars
    color='Management Allocated Cost',
    color_continuous_scale='Viridis'
)

# Improve layout and text
fig.update_traces(
    textposition='outside',
    textfont=dict(size=10),
    hovertemplate="<b>Customer:</b> %{y}<br>Allocated Cost: $%{x:,.2f}<extra></extra>"
)

fig.update_layout(
    height=800,
    width=900,
    title_font=dict(size=18),
    xaxis_title="Management Allocated Cost ($)",
    yaxis_title="Customer ID",
    yaxis=dict(categoryorder='total ascending'),   # Ensures highest at top
    coloraxis_colorbar=dict(
        title="Allocated Cost ($)",
        thickness=15
    ),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()

### PRODUCT : WHAT KINDS OF PRODUCT WAREHOUSE HANDLES

In [32]:
product_master_df

,Product ID,Product Size,Product Category,Handling Type
0,P0001,Small,C,Standard
1,P0002,Large,C,Ambient
2,P0003,Large,C,Chilled
3,P0004,Oversize,B,Ambient
4,P0005,Medium,A,Fragile
...,...,...,...,...
495,P0496,Small,A,Chilled
496,P0497,Oversize,B,Ambient
497,P0498,Small,A,Standard
498,P0499,Oversize,C,Ambient


In [33]:
product_master_df.isnull().sum()

Product ID          0
Product Size        0
Product Category    0
Handling Type       0
dtype: int64

In [34]:
product_master_df[['Product Size','Handling Type']]=product_master_df[['Product Size','Handling Type']].astype('object')

In [35]:
product_master_df['Product Size'].unique()

array(['Small', 'Large', 'Oversize', 'Medium'], dtype=object)

In [36]:
product_master_df['Handling Type'].unique()

array(['Standard', 'Ambient', 'Chilled', 'Fragile'], dtype=object)

In [37]:

# Create crosstab
handling_pd = pd.crosstab(
    product_master_df['Handling Type'], 
    product_master_df['Product Size']
)

# Reorder for better visualization (optional but recommended)
size_order = ['Small', 'Medium', 'Large', 'Oversize']
handling_order = ['Standard', 'Ambient', 'Chilled', 'Fragile']

handling_pd = handling_pd.reindex(index=handling_order, columns=size_order)

# Create Heatmap
fig = px.imshow(
    handling_pd,
    text_auto=True,                          
    color_continuous_scale='Viridis',          # Or 'Viridis', 'Plasma', 'Greens'
    aspect="auto",
    title="<b>Product Mix: Handling Type vs Product Size</b><br><sup>Number of Products in Each Category</sup>"
)

fig.update_layout(
    height=550,
    width=750,
    title_font=dict(size=18),
    xaxis_title="Product Size",
    yaxis_title="Handling Type",
    coloraxis_colorbar=dict(
        title="Count of Products",
        thickness=15
    )
)

# Improve text inside heatmap
fig.update_traces(
    textfont=dict(size=14, color="black"),
    hovertemplate="<b>Handling Type:</b> %{y}<br><b>Product Size:</b> %{x}<br><b>Count:</b> %{z}<extra></extra>"
)

fig.show()

##### INFORMATION MISSING IN PRODUCT MASTER IS CUSTOMER ID (WE CANNOT LINK TO EACH CUSTOMER TO CALCULATE COST TO SERVE FOR THEM BASED ON PRODUCT TYPES)

### REVENUE : AMOUNT GENERATED FROM EACH CUSTOMERS

In [38]:
customer_pricing_df

,Customer Id,Activity Type,Unit Type,Contract Rate
0,C001,Storage,pallet_day,8.96
1,C001,Receipt,pallet,3.92
2,C001,Dispatch,order,4.48
3,C001,Pick,pick,0.17
4,C001,Returns,return,11.20
...,...,...,...,...
205,C030,Dispatch,order,4.43
206,C030,Pick,pick,0.14
207,C030,Returns,return,8.54
208,C030,Rework,job,11.43


In [39]:
m1Revenue_df

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312,4.43,1382.16
6238,Month 1,2026-01-31,C030,Pick,688,0.14,96.32
6239,Month 1,2026-01-31,C030,Returns,27,8.54,230.58
6240,Month 1,2026-01-31,C030,Rework,4,11.43,45.72


In [40]:
m1Revenue_df.isnull().sum()

Month               0
Date                0
Customer Id         0
Charge Type         0
Charged Quantity    0
Rate                0
Revenue             0
dtype: int64

##### data integrity check

In [41]:
# ==========================================
# Data Integrity Checks - Revenue Data (Month 1)
# ==========================================

print("=== 1. Checking if all Charge Types exist in Pricing Data ===\n")

# Get unique Charge Types from revenue data
revenue_charge_types = set(m1Revenue_df['Charge Type'].unique())

# Get unique Activity Types from pricing data
pricing_activity_types = set(customer_pricing_df['Activity Type'].unique())

# Check missing charge types
missing_in_pricing = revenue_charge_types - pricing_activity_types
extra_in_pricing = pricing_activity_types - revenue_charge_types

print("Charge Types in Revenue Data but NOT in Pricing Data:", missing_in_pricing if missing_in_pricing else "None")
print("Activity Types in Pricing Data but NOT used in Revenue:", extra_in_pricing if extra_in_pricing else "None")

print("\n=== 2. Checking if Rates match between Revenue and Pricing Data ===\n")

# Merge revenue with pricing to compare rates
rate_check = m1Revenue_df.merge(
    customer_pricing_df[['Customer Id', 'Activity Type', 'Contract Rate']],
    left_on=['Customer Id', 'Charge Type'],
    right_on=['Customer Id', 'Activity Type'],
    how='left'
)

# Find rows where Rate does not match Contract Rate
rate_mismatch = rate_check[rate_check['Rate'] != rate_check['Contract Rate']]

print(f"Total rows with Rate Mismatch: {len(rate_mismatch)}")

if len(rate_mismatch) > 0:
    print("\nSample of mismatched rates:")
    display(rate_mismatch[['Date', 'Customer Id', 'Charge Type', 'Rate', 'Contract Rate']].head(10))
else:
    print("All rates match between Revenue data and Pricing data. Good data quality!")

=== 1. Checking if all Charge Types exist in Pricing Data ===

Charge Types in Revenue Data but NOT in Pricing Data: None
Activity Types in Pricing Data but NOT used in Revenue: None

=== 2. Checking if Rates match between Revenue and Pricing Data ===

Total rows with Rate Mismatch: 29

Sample of mismatched rates:


,Date,Customer Id,Charge Type,Rate,Contract Rate
89,2026-01-01,C014,Rework,12.0,11.54
293,2026-01-02,C014,Rework,12.0,11.54
493,2026-01-03,C014,Rework,12.0,11.54
692,2026-01-04,C014,Rework,12.0,11.54
892,2026-01-05,C014,Rework,12.0,11.54
1095,2026-01-06,C014,Rework,12.0,11.54
1298,2026-01-07,C014,Rework,12.0,11.54
1492,2026-01-08,C014,Rework,12.0,11.54
1693,2026-01-09,C014,Rework,12.0,11.54
2096,2026-01-11,C014,Rework,12.0,11.54


In [42]:
### before we imputed missing as 11.54 lets do back as 12.0
customer_pricing_df[customer_pricing_df['Customer Id']=='C014']

,Customer Id,Activity Type,Unit Type,Contract Rate
91,C014,Storage,pallet_day,8.58
92,C014,Receipt,pallet,3.02
93,C014,Dispatch,order,3.79
94,C014,Pick,pick,0.14
95,C014,Returns,return,9.67
96,C014,Rework,job,11.54
97,C014,Urgent Order,order,13.57


In [43]:
# ==========================================
# Fix Incorrect Imputed Rate for C014 Rework
# ==========================================

# Check current value before fixing
print("Before Fix:")
display(customer_pricing_df[
    (customer_pricing_df['Customer Id'] == 'C014') & 
    (customer_pricing_df['Activity Type'] == 'Rework')
])

# Update the rate to correct value (12.01)
customer_pricing_df.loc[
    (customer_pricing_df['Customer Id'] == 'C014') & 
    (customer_pricing_df['Activity Type'] == 'Rework'),
    'Contract Rate'
] = 12.0

# Verify after fix
print("\nAfter Fix:")
display(customer_pricing_df[
    (customer_pricing_df['Customer Id'] == 'C014') & 
    (customer_pricing_df['Activity Type'] == 'Rework')
])

Before Fix:


,Customer Id,Activity Type,Unit Type,Contract Rate
96,C014,Rework,job,11.54



After Fix:


,Customer Id,Activity Type,Unit Type,Contract Rate
96,C014,Rework,job,12.0


In [44]:
rate_check = m1Revenue_df.merge(
    customer_pricing_df[['Customer Id', 'Activity Type', 'Contract Rate']],
    left_on=['Customer Id', 'Charge Type'],
    right_on=['Customer Id', 'Activity Type'],
    how='left'
)

# Find rows where Rate does not match Contract Rate
rate_mismatch = rate_check[rate_check['Rate'] != rate_check['Contract Rate']]

print(f"Total rows with Rate Mismatch: {len(rate_mismatch)}")

if len(rate_mismatch) > 0:
    print("\nSample of mismatched rates:")
    display(rate_mismatch[['Date', 'Customer Id', 'Charge Type', 'Rate', 'Contract Rate']].head(10))
else:
    print("All rates match between Revenue data and Pricing data")

Total rows with Rate Mismatch: 0
All rates match between Revenue data and Pricing data


In [45]:
m1Revenue_df.isnull().sum()

Month               0
Date                0
Customer Id         0
Charge Type         0
Charged Quantity    0
Rate                0
Revenue             0
dtype: int64

In [46]:
m1Revenue_df

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312,4.43,1382.16
6238,Month 1,2026-01-31,C030,Pick,688,0.14,96.32
6239,Month 1,2026-01-31,C030,Returns,27,8.54,230.58
6240,Month 1,2026-01-31,C030,Rework,4,11.43,45.72


In [47]:
# ==========================================
# Total Revenue per Customer - Month 1
# ==========================================

# Aggregate revenue at customer level
customer_revenue_m1 = m1Revenue_df.groupby('Customer Id').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Transactions=('Revenue', 'count')   # Optional: number of transaction lines
).reset_index()

# Sort by highest revenue
customer_revenue_m1 = customer_revenue_m1.sort_values('Total_Revenue', ascending=False).reset_index(drop=True)

# Add Rank
customer_revenue_m1.insert(0, 'Rank', range(1, len(customer_revenue_m1) + 1))

# Calculate percentage of total revenue
total_revenue_m1 = customer_revenue_m1['Total_Revenue'].sum()
customer_revenue_m1['Revenue_Share_%'] = (customer_revenue_m1['Total_Revenue'] / total_revenue_m1 * 100).round(2)

# Display
print(f"Total Revenue Generated in Month 1: ${total_revenue_m1:,.2f}")
print("\n=== Top 10 Customers by Revenue (Month 1) ===\n")
display(customer_revenue_m1.head(10))

Total Revenue Generated in Month 1: $3,553,039.92

=== Top 10 Customers by Revenue (Month 1) ===



,Rank,Customer Id,Total_Revenue,Total_Transactions,Revenue_Share_%
0,1,C005,441500.81,162,12.43
1,2,C001,199569.99,179,5.62
2,3,C002,169240.47,213,4.76
3,4,C004,154036.11,209,4.34
4,5,C012,120033.01,207,3.38
5,6,C013,115901.17,207,3.26
6,7,C030,114653.57,212,3.23
7,8,C016,113404.74,214,3.19
8,9,C022,110145.97,214,3.10
9,10,C014,106785.08,211,3.01


In [48]:

fig = px.bar(
    customer_revenue_m1,
    x='Total_Revenue',
    y='Customer Id',
    orientation='h',
    title="<b>Total Revenue by Customer - Month 1</b>",
    text='Revenue_Share_%',                 
    color='Total_Revenue',
    color_continuous_scale='Viridis'
)

fig.update_traces(
    texttemplate='%{text:.2f}%',
    textposition='outside',
    textfont=dict(size=9),
    hovertemplate="<b>%{y}</b><br>Revenue: $%{x:,.2f}<br>Share: %{text:.2f}%<extra></extra>"
)

fig.update_layout(
    height=800,
    width=950,
    title_font=dict(size=18),
    xaxis_title="Total Revenue ($)",
    yaxis_title="Customer ID",
    yaxis=dict(categoryorder='total ascending'),
    coloraxis_colorbar=dict(title="Total Revenue ($)"),
    plot_bgcolor='white'
)

fig.show()

### MANAGEMENT COST ALLOCATION BY REVENUE METHOD VS REVENUE GENERATED FROM CUSTOMERS

In [49]:
# ==========================================
# Merge Revenue and Management Allocated Cost
# ==========================================

profitability_view = customer_revenue_m1[['Customer Id', 'Total_Revenue']].merge(
    management_allocation_df_m1[['Customer Id', 'Management Allocated Cost']],
    on='Customer Id',
    how='left'
)

# Sort by Revenue (highest first)
profitability_view = profitability_view.sort_values('Total_Revenue', ascending=False).reset_index(drop=True)

print("=== Revenue vs Management Allocated Cost ===\n")
display(profitability_view)

=== Revenue vs Management Allocated Cost ===



,Customer Id,Total_Revenue,Management Allocated Cost
0,C005,441500.81,16464.452593
1,C001,199569.99,7442.366050
2,C002,169240.47,6311.317289
3,C004,154036.11,5744.316145
4,C012,120033.01,4476.272202
5,C013,115901.17,4322.187583
6,C030,114653.57,4275.662072
7,C016,113404.74,4229.090691
8,C022,110145.97,4107.564608
9,C014,106785.08,3982.230264


In [50]:
total_revenue = profitability_view['Total_Revenue'].sum()

profitability_view['Revenue_Share_%'] = (
    profitability_view['Total_Revenue'] / total_revenue
) * 100
profitability_view

,Customer Id,Total_Revenue,Management Allocated Cost,Revenue_Share_%
0,C005,441500.81,16464.452593,12.426002
1,C001,199569.99,7442.366050,5.616880
2,C002,169240.47,6311.317289,4.763258
3,C004,154036.11,5744.316145,4.335333
4,C012,120033.01,4476.272202,3.378319
5,C013,115901.17,4322.187583,3.262028
6,C030,114653.57,4275.662072,3.226915
7,C016,113404.74,4229.090691,3.191767
8,C022,110145.97,4107.564608,3.100049
9,C014,106785.08,3982.230264,3.005457


In [51]:
import plotly.express as px

fig = px.scatter(
    profitability_view,
    x='Total_Revenue',
    y='Management Allocated Cost',
    hover_name='Customer Id',
    trendline='ols',                 
    title="<b>Current Method :Management Allocated Cost vs Customer Revenue</b><br><sup>Do higher revenue customers get higher cost allocation?</sup>",
    labels={
        'Total_Revenue': 'Total Revenue ($)',
        'Management Allocated Cost': 'Management Allocated Cost ($)'
    }
)

fig.update_traces(
    marker=dict(size=10, color='#E63946'),
    hovertemplate="<b>%{hovertext}</b><br>Revenue: $%{x:,.2f}<br>Allocated Cost: $%{y:,.2f}<extra></extra>"
)

fig.update_layout(
    height=600,
    width=850,
    title_font=dict(size=18),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()

##### shows linear relationship (Highest Revenue Customer has higher Cost) : current method

In [52]:
correlation = profitability_view['Total_Revenue'].corr(profitability_view['Management Allocated Cost'])
print(f"Correlation between Revenue and Management Allocated Cost: {correlation:.3f}")

Correlation between Revenue and Management Allocated Cost: 1.000


##### Current Managment Strategy
##### Whoever generates more revenue carries more of the management cost

In [53]:
# ==========================================
# Customer Profitability View (Estimated)
# ==========================================

# Merge Revenue and Management Allocated Cost
profit_view = customer_revenue_m1[['Customer Id', 'Total_Revenue']].merge(
    management_allocation_df_m1[['Customer Id', 'Management Allocated Cost']],
    on='Customer Id',
    how='left'
)

# Calculate Estimated Profit and Margin
profit_view['Estimated_Profit'] = profit_view['Total_Revenue'] - profit_view['Management Allocated Cost']
profit_view['Profit_Margin_%'] = (profit_view['Estimated_Profit'] / profit_view['Total_Revenue'] * 100).round(2)

# Add Rank
profit_view = profit_view.sort_values('Total_Revenue', ascending=False).reset_index(drop=True)
profit_view.insert(0, 'Rank', range(1, len(profit_view) + 1))

# Reorder columns for better readability
profit_view = profit_view[['Rank', 'Customer Id', 'Total_Revenue', 
                           'Management Allocated Cost', 'Estimated_Profit', 'Profit_Margin_%']]

print("=== Customer Profitability View (Month 1) ===\n")
display(profit_view)

=== Customer Profitability View (Month 1) ===



,Rank,Customer Id,Total_Revenue,Management Allocated Cost,Estimated_Profit,Profit_Margin_%
0,1,C005,441500.81,16464.452593,425036.357407,96.27
1,2,C001,199569.99,7442.366050,192127.623950,96.27
2,3,C002,169240.47,6311.317289,162929.152711,96.27
3,4,C004,154036.11,5744.316145,148291.793855,96.27
4,5,C012,120033.01,4476.272202,115556.737798,96.27
5,6,C013,115901.17,4322.187583,111578.982417,96.27
6,7,C030,114653.57,4275.662072,110377.907928,96.27
7,8,C016,113404.74,4229.090691,109175.649309,96.27
8,9,C022,110145.97,4107.564608,106038.405392,96.27
9,10,C014,106785.08,3982.230264,102802.849736,96.27


management is allocating cost based on revenue percentage, the profit margin becomes almost identical for every customer.
This creates a false picture:

A customer doing $441,500 revenue looks 96.27% profitable.
A customer doing only $85,377 revenue also looks 96.27% profitable.
In reality, their cost to serve is very different, but the current method hides that difference.

Problems with This Method:

It hides real profitability differences — All customers look equally good (or bad) in terms of margin.
It punishes high-revenue customers by loading them with higher costs.
It doesn’t reflect actual resource usage — A customer using more warehouse space, labor, or management time should logically carry more cost, not just because they have high revenue.
It makes decision-making difficult — You cannot easily identify which customers are truly valuable.

In [54]:
m1Revenue_df

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312,4.43,1382.16
6238,Month 1,2026-01-31,C030,Pick,688,0.14,96.32
6239,Month 1,2026-01-31,C030,Returns,27,8.54,230.58
6240,Month 1,2026-01-31,C030,Rework,4,11.43,45.72


In [55]:
customer_1=m1Revenue_df[m1Revenue_df['Customer Id']=='C001'].copy()
customer_1

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6040,Month 1,2026-01-31,C001,Receipt,27,3.92,105.84
6041,Month 1,2026-01-31,C001,Dispatch,34,4.48,152.32
6042,Month 1,2026-01-31,C001,Pick,842,0.17,143.14
6043,Month 1,2026-01-31,C001,Returns,1,11.20,11.20


In [56]:
def check_customer_date_completeness(df):
    customers = df['Customer Id'].unique()
    missing_info = {}

    for cust in customers:
        cust_df = df[df['Customer Id'] == cust]
        full_range = pd.date_range(start=cust_df['Date'].min(), 
                                   end=cust_df['Date'].max(), 
                                   freq='D')
        missing = full_range.difference(cust_df['Date'])
        
        if not missing.empty:
            missing_info[cust] = missing

    if missing_info:
        print("Customers with missing dates:")
        for cust, dates in missing_info.items():
            print(f"{cust}: {len(dates)} missing dates")
    else:
        print("All customers have complete daily data for Month 1 (no missing dates).")

    return missing_info

# Run the check
missing_dates = check_customer_date_completeness(m1Revenue_df)

All customers have complete daily data for Month 1 (no missing dates).


In [57]:

# ==========================================
# Faceted Daily Revenue Trend - Black Theme
# ==========================================

fig = px.area(
    m1Revenue_df,
    x='Date',
    y='Revenue',
    color='Customer Id',
    facet_col='Customer Id',
    facet_col_wrap=5,
    title="<b>Daily Revenue Trend by Customer - Month 1</b>",
    labels={'Revenue': 'Daily Revenue ($)'}
)

fig.update_traces(
    hovertemplate="<b>%{x|%d %b}</b><br>Revenue: $%{y:,.2f}<extra></extra>"
)

fig.update_layout(
    height=4000,
    width=3000,
    title_font=dict(size=20, color='white'),
    showlegend=False,
    plot_bgcolor='black',          
    paper_bgcolor='black',          
    font=dict(color='white')        
)

# Grid Lines (lighter color for visibility on black)
fig.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#444444',           
    color='white'                   
)

fig.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#444444',
    color='white'
)

# Clean facet titles
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [58]:


# ==========================================
# Prepare Weekly Data
# ==========================================

weekly_df = m1Revenue_df.copy()
weekly_df['Week'] = weekly_df['Date'].dt.to_period('W').astype(str)

weekly_revenue = weekly_df.groupby(['Customer Id', 'Week'])['Revenue'].sum().reset_index()
weekly_revenue = weekly_revenue.sort_values(['Customer Id', 'Week'])

# ==========================================
# Faceted Weekly Revenue Trend - Black Theme
# ==========================================

fig = px.area(
    weekly_revenue,
    x='Week',
    y='Revenue',
    color='Customer Id',
    facet_col='Customer Id',
    facet_col_wrap=5,
    title="<b>Weekly Revenue Trend by Customer - Month 1</b>",
    labels={'Revenue': 'Weekly Revenue ($)'}
)

fig.update_traces(
    hovertemplate="<b>Week:</b> %{x}<br>Revenue: $%{y:,.2f}<extra></extra>"
)

fig.update_layout(
    height=1600,
    width=1400,
    title_font=dict(size=20, color='white'),
    showlegend=False,
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white')
)

# Grid Lines
fig.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#444444',
    color='white'
)

fig.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#444444',
    color='white'
)

# Clean facet titles
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [59]:
m1Revenue_df

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312,4.43,1382.16
6238,Month 1,2026-01-31,C030,Pick,688,0.14,96.32
6239,Month 1,2026-01-31,C030,Returns,27,8.54,230.58
6240,Month 1,2026-01-31,C030,Rework,4,11.43,45.72


In [60]:
# ==========================================
# Revenue Mix by Charge Type per Customer
# ==========================================

# Calculate total revenue per customer per charge type
revenue_mix = m1Revenue_df.groupby(['Customer Id', 'Charge Type'])['Revenue'].sum().reset_index()

# Calculate total revenue per customer
total_revenue_per_customer = m1Revenue_df.groupby('Customer Id')['Revenue'].sum().reset_index()
total_revenue_per_customer.columns = ['Customer Id', 'Total_Revenue']

# Merge and calculate percentage
revenue_mix = revenue_mix.merge(total_revenue_per_customer, on='Customer Id')
revenue_mix['Revenue_Share_%'] = (revenue_mix['Revenue'] / revenue_mix['Total_Revenue'] * 100).round(2)

print("Revenue Mix Data:")
display(revenue_mix.head(10))

Revenue Mix Data:


,Customer Id,Charge Type,Revenue,Total_Revenue,Revenue_Share_%
0,C001,Dispatch,11822.72,199569.99,5.92
1,C001,Pick,2840.87,199569.99,1.42
2,C001,Receipt,4476.64,199569.99,2.24
3,C001,Returns,806.40,199569.99,0.40
4,C001,Rework,215.04,199569.99,0.11
5,C001,Storage,179173.12,199569.99,89.78
6,C001,Urgent Order,235.20,199569.99,0.12
7,C002,Dispatch,57392.40,169240.47,33.91
8,C002,Pick,15903.00,169240.47,9.40
9,C002,Receipt,17441.97,169240.47,10.31


In [61]:

# Sort customers by total revenue (highest on top)
customer_order = total_revenue_per_customer.sort_values('Total_Revenue', ascending=False)['Customer Id'].tolist()

fig = px.bar(
    revenue_mix,
    x='Revenue_Share_%',
    y='Customer Id',
    color='Charge Type',
    orientation='h',
    title="<b>Revenue Mix by Charge Type per Customer - Month 1</b>",
    category_orders={'Customer Id': customer_order},
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_traces(
    hovertemplate="<b>%{y}</b><br>%{fullData.name}: %{x:.2f}%<extra></extra>"
)

fig.update_layout(
    height=900,
    width=1000,
    title_font=dict(size=18),
    xaxis_title="Revenue Share (%)",
    yaxis_title="Customer ID",
    barmode='stack',
    plot_bgcolor='grey',
    paper_bgcolor='grey',
    legend_title="Charge Type"
)

fig.show()

In [62]:
m1Revenue_df.head()

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60


In [63]:
revenue_by_activity = (
    m1Revenue_df
    .groupby('Charge Type')
    .agg(
        Quantity=('Charged Quantity','sum'),
        Revenue=('Revenue','sum')
    )
    .reset_index()
)

revenue_by_activity

,Charge Type,Quantity,Revenue
0,Dispatch,132890,508004.18
1,Pick,961653,137829.55
2,Receipt,58237,196185.27
3,Returns,16417,159046.73
4,Rework,11198,130880.04
5,Storage,271319,2289819.88
6,Urgent Order,8877,131274.27


In [64]:
revenue_by_activity['Revenue per Unit'] = (
    revenue_by_activity['Revenue']
    / revenue_by_activity['Quantity']
)

display(revenue_by_activity)

,Charge Type,Quantity,Revenue,Revenue per Unit
0,Dispatch,132890,508004.18,3.822742
1,Pick,961653,137829.55,0.143326
2,Receipt,58237,196185.27,3.368739
3,Returns,16417,159046.73,9.687929
4,Rework,11198,130880.04,11.687805
5,Storage,271319,2289819.88,8.439585
6,Urgent Order,8877,131274.27,14.788135


In [65]:
revenue_by_activity.sort_values(ascending=False,by='Quantity')[['Charge Type','Quantity']]

,Charge Type,Quantity
1,Pick,961653
5,Storage,271319
0,Dispatch,132890
2,Receipt,58237
3,Returns,16417
4,Rework,11198
6,Urgent Order,8877


In [66]:
revenue_by_activity.sort_values(ascending=False,by='Revenue')[['Charge Type','Revenue']]

,Charge Type,Revenue
5,Storage,2289819.88
0,Dispatch,508004.18
2,Receipt,196185.27
3,Returns,159046.73
1,Pick,137829.55
6,Urgent Order,131274.27
4,Rework,130880.04


In [67]:
revenue_by_activity.sort_values(ascending=False,by='Revenue per Unit')[['Charge Type','Revenue per Unit']]

,Charge Type,Revenue per Unit
6,Urgent Order,14.788135
4,Rework,11.687805
3,Returns,9.687929
5,Storage,8.439585
0,Dispatch,3.822742
2,Receipt,3.368739
1,Pick,0.143326


##### Pick is the highest-volume activity but contributes relatively little revenue per transaction.
##### Storage is the largest revenue contributor.
##### Operational volume and revenue generation are not strongly correlated.
##### Urgent Orders have the highest revenue yield per transaction..Urgent Orders generate substantially more revenue per transaction

In [68]:
revenue_by_activity['Revenue %'] = (
    revenue_by_activity['Revenue']
    / revenue_by_activity['Revenue'].sum()
    * 100
)

revenue_by_activity.sort_values('Revenue', ascending=False)

,Charge Type,Quantity,Revenue,Revenue per Unit,Revenue %
5,Storage,271319,2289819.88,8.439585,64.446782
0,Dispatch,132890,508004.18,3.822742,14.297734
2,Receipt,58237,196185.27,3.368739,5.521617
3,Returns,16417,159046.73,9.687929,4.476356
1,Pick,961653,137829.55,0.143326,3.879201
6,Urgent Order,8877,131274.27,14.788135,3.694703
4,Rework,11198,130880.04,11.687805,3.683607


In [69]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Sorting
df_qty = revenue_by_activity.sort_values(by='Quantity', ascending=False)
df_rev = revenue_by_activity.sort_values(by='Revenue', ascending=False)
df_rpu = revenue_by_activity.sort_values(by='Revenue per Unit', ascending=False)
df_pct = revenue_by_activity.sort_values(by='Revenue %', ascending=False)

# Dark theme dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Operational Volume",
        "Total Revenue",
        "Revenue per Unit",
        "Revenue Contribution %"
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# Helper function for hover
def hover_template(label):
    return label + ": %{y:,.2f}<extra></extra>"

# 1. Quantity
fig.add_trace(
    go.Bar(
        x=df_qty['Charge Type'],
        y=df_qty['Quantity'],
        marker=dict(
            color=df_qty['Quantity'],
            colorscale='Blues',
            line=dict(width=0.5, color='white')
        ),
        hovertemplate=hover_template("Quantity")
    ),
    row=1, col=1
)

# 2. Revenue
fig.add_trace(
    go.Bar(
        x=df_rev['Charge Type'],
        y=df_rev['Revenue'],
        marker=dict(
            color=df_rev['Revenue'],
            colorscale='Greens',
            line=dict(width=0.5, color='white')
        ),
        hovertemplate=hover_template("Revenue")
    ),
    row=1, col=2
)

# 3. Revenue per Unit
fig.add_trace(
    go.Bar(
        x=df_rpu['Charge Type'],
        y=df_rpu['Revenue per Unit'],
        marker=dict(
            color=df_rpu['Revenue per Unit'],
            colorscale='Plasma',
            line=dict(width=0.5, color='white')
        ),
        hovertemplate=hover_template("Rev/Unit")
    ),
    row=2, col=1
)

# 4. Revenue %
fig.add_trace(
    go.Bar(
        x=df_pct['Charge Type'],
        y=df_pct['Revenue %'],
        marker=dict(
            color=df_pct['Revenue %'],
            colorscale='Inferno',
            line=dict(width=0.5, color='white')
        ),
        hovertemplate=hover_template("Revenue %")
    ),
    row=2, col=2
)

# Layout (this is where it becomes "sexy")
fig.update_layout(
    title=dict(
        text="Activity Performance Dashboard (Volume vs Revenue vs Efficiency) for Month 1",
        x=0.5,
        font=dict(size=22, color="white")
    ),
    paper_bgcolor="#0e1117",
    plot_bgcolor="#0e1117",
    font=dict(color="white"),
    height=850,
    showlegend=False,
    transition=dict(duration=800)
)

fig.update_xaxes(tickangle=35, gridcolor="rgba(255,255,255,0.05)")
fig.update_yaxes(gridcolor="rgba(255,255,255,0.05)")

fig.show()

##### The company is primarily being paid for holding inventory, not for moving inventory.

Storage is the dominant revenue-generating activity, contributing 64.4% of total revenue. In contrast, Pick activities account for the highest operational volume (961,653 transactions) but generate only 3.9% of revenue, indicating a potentially low revenue yield relative to workload. Urgent Orders and Rework activities generate the highest revenue per transaction despite low volumes, suggesting that specialised services may provide greater revenue leverage than routine warehouse operations. Further cost allocation analysis is required to determine which activities ultimately create or destroy value.

### ACTIVITIES: WHAT ACTIVITIES ARE CARRIED OUT IN WAREHOUSE

In [70]:
m1Activities_df

,Month,Date,Customer Id,Activity Type,Quantity,Data Quality Note
0,Month 1,2026-01-01,C001,Storage,913.0,NaN
1,Month 1,2026-01-01,C001,Receipt,33.0,NaN
2,Month 1,2026-01-01,C001,Dispatch,54.0,NaN
3,Month 1,2026-01-01,C001,Pick,231.0,NaN
4,Month 1,2026-01-01,C001,Returns,3.0,NaN
...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0,NaN
6238,Month 1,2026-01-31,C030,Pick,688.0,NaN
6239,Month 1,2026-01-31,C030,Returns,27.0,NaN
6240,Month 1,2026-01-31,C030,Rework,4.0,NaN


In [71]:
m1Activities_df.drop('Data Quality Note',axis=1,inplace=True)
m1Activities_df.isnull().sum()

Month            0
Date             0
Customer Id      0
Activity Type    0
Quantity         2
dtype: int64

In [72]:
m1Activities_df.head()

,Month,Date,Customer Id,Activity Type,Quantity
0,Month 1,2026-01-01,C001,Storage,913.0
1,Month 1,2026-01-01,C001,Receipt,33.0
2,Month 1,2026-01-01,C001,Dispatch,54.0
3,Month 1,2026-01-01,C001,Pick,231.0
4,Month 1,2026-01-01,C001,Returns,3.0


In [73]:
m1Activities_df[m1Activities_df['Quantity'].isnull()]

,Month,Date,Customer Id,Activity Type,Quantity
1072,Month 1,2026-01-06,C011,Pick,NaN
1276,Month 1,2026-01-07,C011,Pick,NaN


In [74]:
##### DATA INTEGIRTY CHECK : charge type vs activity type uniques 

set(m1Revenue_df['Charge Type'])

{'Dispatch', 'Pick', 'Receipt', 'Returns', 'Rework', 'Storage', 'Urgent Order'}

In [75]:
set(m1Activities_df['Activity Type'])

{'Dispatch', 'Pick', 'Receipt', 'Returns', 'Rework', 'Storage', 'Urgent Order'}

In [76]:
set(m1Revenue_df['Charge Type'])==set(m1Activities_df['Activity Type'])

True

In [77]:
#####  DATA INTEGIRTY CHECK : charge type vs activity type for each row
activity_check = m1Activities_df.merge(
    m1Revenue_df,
    left_on=['Date', 'Customer Id', 'Activity Type'],
    right_on=['Date', 'Customer Id', 'Charge Type'],
    how='left',
    indicator=True
)
activity_check

,Month_x,Date,Customer Id,Activity Type,Quantity,Month_y,Charge Type,Charged Quantity,Rate,Revenue,_merge
0,Month 1,2026-01-01,C001,Storage,913.0,Month 1,Storage,913,8.96,8180.48,both
1,Month 1,2026-01-01,C001,Receipt,33.0,Month 1,Receipt,33,3.92,129.36,both
2,Month 1,2026-01-01,C001,Dispatch,54.0,Month 1,Dispatch,54,4.48,241.92,both
3,Month 1,2026-01-01,C001,Pick,231.0,Month 1,Pick,231,0.17,39.27,both
4,Month 1,2026-01-01,C001,Returns,3.0,Month 1,Returns,3,11.20,33.60,both
...,...,...,...,...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0,Month 1,Dispatch,312,4.43,1382.16,both
6238,Month 1,2026-01-31,C030,Pick,688.0,Month 1,Pick,688,0.14,96.32,both
6239,Month 1,2026-01-31,C030,Returns,27.0,Month 1,Returns,27,8.54,230.58,both
6240,Month 1,2026-01-31,C030,Rework,4.0,Month 1,Rework,4,11.43,45.72,both


In [78]:
activity_check['_merge'].value_counts()

_merge
both          6242
left_only        0
right_only       0
Name: count, dtype: int64

All 6,242 activity records have a corresponding billing activity type in the revenue dataset. No activity records were found without a matching charge type, suggesting consistency between operational activity classifications and revenue classifications

##### Was the same quantity billed as was operationally recorded?

In [79]:
len(m1Activities_df),len(activity_check)

(6242, 6242)

In [80]:
activity_check['qty_match'] = (
    activity_check['Quantity'] ==
    activity_check['Charged Quantity']
)
activity_check

,Month_x,Date,Customer Id,Activity Type,Quantity,Month_y,Charge Type,Charged Quantity,Rate,Revenue,_merge,qty_match
0,Month 1,2026-01-01,C001,Storage,913.0,Month 1,Storage,913,8.96,8180.48,both,True
1,Month 1,2026-01-01,C001,Receipt,33.0,Month 1,Receipt,33,3.92,129.36,both,True
2,Month 1,2026-01-01,C001,Dispatch,54.0,Month 1,Dispatch,54,4.48,241.92,both,True
3,Month 1,2026-01-01,C001,Pick,231.0,Month 1,Pick,231,0.17,39.27,both,True
4,Month 1,2026-01-01,C001,Returns,3.0,Month 1,Returns,3,11.20,33.60,both,True
...,...,...,...,...,...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0,Month 1,Dispatch,312,4.43,1382.16,both,True
6238,Month 1,2026-01-31,C030,Pick,688.0,Month 1,Pick,688,0.14,96.32,both,True
6239,Month 1,2026-01-31,C030,Returns,27.0,Month 1,Returns,27,8.54,230.58,both,True
6240,Month 1,2026-01-31,C030,Rework,4.0,Month 1,Rework,4,11.43,45.72,both,True


In [81]:
activity_check['qty_match'].value_counts()

qty_match
True     5891
False     351
Name: count, dtype: int64

Out of all activity records, 351 rows have a mismatch between Activity Quantity and Charged Quantity.

In [82]:
mismatches = activity_check[activity_check['qty_match'] == False]
display(
    mismatches[
        [
            'Date',
            'Customer Id',
            'Activity Type',
            'Quantity',
            'Charged Quantity'
        ]
    ].head(20)
)

,Date,Customer Id,Activity Type,Quantity,Charged Quantity
14,2026-01-01,C003,Dispatch,83.0,54
15,2026-01-01,C003,Pick,3802.0,2028
17,2026-01-01,C003,Rework,6.0,2
23,2026-01-01,C004,Returns,102.0,73
24,2026-01-01,C004,Rework,66.0,55
25,2026-01-01,C004,Urgent Order,32.0,25
47,2026-01-01,C008,Dispatch,98.0,86
70,2026-01-01,C012,Storage,279.0,270
85,2026-01-01,C014,Receipt,83.0,78
111,2026-01-01,C017,Urgent Order,1.0,0


In [83]:
mismatches['Activity Type'].value_counts()

Activity Type
Rework          87
Pick            58
Dispatch        55
Returns         52
Urgent Order    49
Receipt         30
Storage         20
Name: count, dtype: int64

In [84]:
mismatches['Customer Id'].unique()

<ArrowStringArray>
['C003', 'C004', 'C008', 'C012', 'C014', 'C017', 'C027', 'C030', 'C001',
 'C010', 'C026', 'C011', 'C013', 'C018', 'C021', 'C025', 'C006', 'C023',
 'C005', 'C016', 'C020', 'C009', 'C019', 'C022', 'C024', 'C029', 'C015',
 'C028', 'C002', 'C007']
Length: 30, dtype: str

In [85]:
activity_check.isnull().sum()

Month_x             0
Date                0
Customer Id         0
Activity Type       0
Quantity            2
Month_y             0
Charge Type         0
Charged Quantity    0
Rate                0
Revenue             0
_merge              0
qty_match           0
dtype: int64

In [86]:
mismatches['diff'] = mismatches['Quantity'] - mismatches['Charged Quantity']
underbilling = mismatches[mismatches['diff'] > 0]
overbilling = mismatches[mismatches['diff'] < 0]
exact_match = mismatches[mismatches['diff'] == 0]


print("Underbilling (revenue loss):", len(underbilling))
print("Overbilling (revenue gain):", len(overbilling))
print("Exact zero diff:", len(exact_match))

Underbilling (revenue loss): 349
Overbilling (revenue gain): 0
Exact zero diff: 0


##### Operational work was done (activities exist)But billing recorded LESS than actual work. No cases where customers were overcharged
##### The revenue system consistently under-bills operational activity. All 349 mismatches show Activity Quantity exceeding Charged Quantity, indicating systematic revenue leakage rather than random billing errors or overcharging.

In [87]:
mismatches['qty_gap'] = mismatches['Quantity'] - mismatches['Charged Quantity']
mismatches['revenue_leakage'] = mismatches['qty_gap'] * mismatches['Rate']
total_leakage = mismatches['revenue_leakage'].sum()

print("Total Revenue Leakage: $", total_leakage)

Total Revenue Leakage: $ 40802.61


In [88]:
leakage_by_activity = (
    mismatches
    .groupby('Activity Type')['revenue_leakage']
    .sum()
    .sort_values(ascending=False)
)

display(leakage_by_activity)

Activity Type
Dispatch        16228.84
Returns          5787.67
Pick             5416.35
Rework           5072.99
Urgent Order     4221.78
Storage          3707.56
Receipt           367.42
Name: revenue_leakage, dtype: float64

high operational volume ≠ high financial error risk

The organization experiences systematic revenue leakage of approximately 16.2k in Dispatch activities alone, making it the largest contributor to underbilling. While Storage drives the majority of revenue, its billing accuracy remains relatively stable. Overall, leakage is concentrated in operational movement activities rather than storage or pricing-intensive services.

In [89]:
mismatches['qty_gap'] = (
    mismatches['Quantity'] - mismatches['Charged Quantity']
)
mismatches_sorted = mismatches.sort_values(
    by='qty_gap',
    ascending=False
)


activity_mismatch_summary = (
    mismatches
    .groupby('Activity Type')
    .agg(
        error_count=('qty_gap', 'count'),
        total_gap=('qty_gap', 'sum'),
    )
    .sort_values(by='total_gap', ascending=False)
)
activity_mismatch_summary


,error_count,total_gap
Activity Type,,
Pick,56,33921.0
Dispatch,55,3884.0
Returns,52,674.0
Rework,87,466.0
Storage,20,452.0
Urgent Order,49,326.0
Receipt,30,108.0


In [90]:
mismatches['Activity Type'].value_counts()

Activity Type
Rework          87
Pick            58
Dispatch        55
Returns         52
Urgent Order    49
Receipt         30
Storage         20
Name: count, dtype: int64

In [91]:
m1Activities_df

,Month,Date,Customer Id,Activity Type,Quantity
0,Month 1,2026-01-01,C001,Storage,913.0
1,Month 1,2026-01-01,C001,Receipt,33.0
2,Month 1,2026-01-01,C001,Dispatch,54.0
3,Month 1,2026-01-01,C001,Pick,231.0
4,Month 1,2026-01-01,C001,Returns,3.0
...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0
6238,Month 1,2026-01-31,C030,Pick,688.0
6239,Month 1,2026-01-31,C030,Returns,27.0
6240,Month 1,2026-01-31,C030,Rework,4.0


##### Compare Activity DF vs Revenue DF (Volume reality check)

In [92]:
activity_volume = (
    m1Activities_df
    .groupby('Activity Type')['Quantity']
    .sum()
    .sort_values(ascending=False)
)

activity_volume_df = activity_volume.reset_index()
activity_volume_df.columns = ['Activity Type', 'Quantity']
activity_volume_df


,Activity Type,Quantity
0,Pick,993588.0
1,Storage,271771.0
2,Dispatch,136774.0
3,Receipt,58345.0
4,Returns,17091.0
5,Rework,11664.0
6,Urgent Order,9203.0


In [93]:
revenue_volume = revenue_by_activity[['Charge Type', 'Quantity']].sort_values(ascending=False,by='Quantity')
revenue_volume

,Charge Type,Quantity
1,Pick,961653
5,Storage,271319
0,Dispatch,132890
2,Receipt,58237
3,Returns,16417
4,Rework,11198
6,Urgent Order,8877


In [94]:
activity_volume_df = activity_volume_df.rename(
    columns={'Activity Type': 'Charge Type'}
)
activity_volume_df

,Charge Type,Quantity
0,Pick,993588.0
1,Storage,271771.0
2,Dispatch,136774.0
3,Receipt,58345.0
4,Returns,17091.0
5,Rework,11664.0
6,Urgent Order,9203.0


In [95]:
volume_compare = activity_volume_df.merge(
    revenue_volume,
    on='Charge Type',
    how='inner',
    suffixes=('_Activity', '_Revenue')
)
volume_compare['diff'] = (
    volume_compare['Quantity_Activity']
    - volume_compare['Quantity_Revenue']
)
volume_compare.sort_values(
    by='Quantity_Activity',
    ascending=False
)

,Charge Type,Quantity_Activity,Quantity_Revenue,diff
0,Pick,993588.0,961653,31935.0
1,Storage,271771.0,271319,452.0
2,Dispatch,136774.0,132890,3884.0
3,Receipt,58345.0,58237,108.0
4,Returns,17091.0,16417,674.0
5,Rework,11664.0,11198,466.0
6,Urgent Order,9203.0,8877,326.0


Despite 351 quantity-level mismatches between Activity and Revenue datasets, the relative ranking and magnitude of operational activities remain highly consistent across both sources. This indicates that while billing discrepancies exist at transaction level, they do not materially distort the overall operational profile of the business. Pick and Storage remain the dominant drivers of warehouse activity in both datasets.

In [96]:
gap_in_quantity=activity_mismatch_summary.copy()
gap_in_quantity

,error_count,total_gap
Activity Type,,
Pick,56,33921.0
Dispatch,55,3884.0
Returns,52,674.0
Rework,87,466.0
Storage,20,452.0
Urgent Order,49,326.0
Receipt,30,108.0


In [97]:
gap_in_quantity.reset_index(inplace=True)
gap_in_quantity

,Activity Type,error_count,total_gap
0,Pick,56,33921.0
1,Dispatch,55,3884.0
2,Returns,52,674.0
3,Rework,87,466.0
4,Storage,20,452.0
5,Urgent Order,49,326.0
6,Receipt,30,108.0


In [98]:
gap_in_quantity=gap_in_quantity.rename(columns={'Activity Type':'Charge Type'})
gap_in_quantity

,Charge Type,error_count,total_gap
0,Pick,56,33921.0
1,Dispatch,55,3884.0
2,Returns,52,674.0
3,Rework,87,466.0
4,Storage,20,452.0
5,Urgent Order,49,326.0
6,Receipt,30,108.0


In [99]:
##### ARE THOSE DIFFERENCE FROM THOSE 351 TRANSCATION (DIFFERENCE IN QUANTITY IN ACTIVITY WITH CHARGED QUANTITY IN REVENUE)

check_351=volume_compare.merge(gap_in_quantity,on='Charge Type',how='inner')
check_351

,Charge Type,Quantity_Activity,Quantity_Revenue,diff,error_count,total_gap
0,Pick,993588.0,961653,31935.0,56,33921.0
1,Storage,271771.0,271319,452.0,20,452.0
2,Dispatch,136774.0,132890,3884.0,55,3884.0
3,Receipt,58345.0,58237,108.0,30,108.0
4,Returns,17091.0,16417,674.0,52,674.0
5,Rework,11664.0,11198,466.0,87,466.0
6,Urgent Order,9203.0,8877,326.0,49,326.0


In [100]:
check_351['gap diff']=check_351['diff']-check_351['total_gap']
check_351[['Charge Type','gap diff']]

,Charge Type,gap diff
0,Pick,-1986.0
1,Storage,0.0
2,Dispatch,0.0
3,Receipt,0.0
4,Returns,0.0
5,Rework,0.0
6,Urgent Order,0.0


The activity and revenue datasets are highly consistent. Nearly all quantity discrepancies are fully explained by the 351 row-level mismatches, with only a minor residual discrepancy in Pick activity attributable to missing or null records. No structural misalignment exists across activity types, confirming strong data integrity for downstream profitability analysis.

In [101]:

highest_vol_activity=m1Activities_df.groupby('Activity Type')['Quantity'].sum().reset_index().sort_values(ascending=False,by='Quantity')


fig = px.bar(
    highest_vol_activity.sort_values('Quantity', ascending=True),
    x='Quantity',
    y='Activity Type',
    orientation='h',
    title="<b>Total Quantity Processed by Activity Type - Month 1</b>",
    text='Quantity',
    color='Activity Type'
)

fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside',
    textfont=dict(size=12, color='black')
)

fig.update_layout(
    height=550,
    showlegend=False,
    xaxis_title="Total Quantity Processed",
    yaxis_title="Activity Type",
    plot_bgcolor='white'
)

fig.show()

The analysis identifies a systematic underbilling issue where operational activity exceeds billed quantity in 351 transactions. This indicates a consistent gap between warehouse execution records and revenue capture, requiring management attention to strengthen billing integrity and improve revenue assurance processes.

In [102]:
### WE HAVE REVENUE FROM EACH CUSTOMER, WE HAVE MANAGMENT CURRENT COST ALLOCATION TO CUSTOMER 
##### SO WHOM MANAGEMENT CURRENTLY THINKS PROFITABLE


management_allocation_df_m1

,Month,Customer Id,Allocation Method,Management Allocated Cost,Overall_Cost_Percentage
0,Month 1,C001,Revenue percentage,7442.366050,5.62
1,Month 1,C002,Revenue percentage,6311.317289,4.76
2,Month 1,C003,Revenue percentage,3530.021385,2.66
3,Month 1,C004,Revenue percentage,5744.316145,4.34
4,Month 1,C005,Revenue percentage,16464.452593,12.43
5,Month 1,C006,Revenue percentage,3384.693677,2.55
6,Month 1,C007,Revenue percentage,3183.881424,2.40
7,Month 1,C008,Revenue percentage,3674.910167,2.77
8,Month 1,C009,Revenue percentage,3915.559982,2.96
9,Month 1,C010,Revenue percentage,3190.000297,2.41


In [103]:
total_revenue_per_customer

,Customer Id,Total_Revenue
0,C001,199569.99
1,C002,169240.47
2,C003,94658.92
3,C004,154036.11
4,C005,441500.81
5,C006,90761.90
6,C007,85377.04
7,C008,98544.17
8,C009,104997.29
9,C010,85541.12


In [104]:
total_revenue_per_customer=total_revenue_per_customer.sort_values(ascending=False,by='Total_Revenue')
total_revenue_per_customer

,Customer Id,Total_Revenue
4,C005,441500.81
0,C001,199569.99
1,C002,169240.47
3,C004,154036.11
11,C012,120033.01
12,C013,115901.17
29,C030,114653.57
15,C016,113404.74
21,C022,110145.97
13,C014,106785.08


In [105]:
managment_current_profitable=management_allocation_df_m1.merge(total_revenue_per_customer,on='Customer Id',how='inner')
managment_current_profitable

,Month,Customer Id,Allocation Method,Management Allocated Cost,Overall_Cost_Percentage,Total_Revenue
0,Month 1,C001,Revenue percentage,7442.366050,5.62,199569.99
1,Month 1,C002,Revenue percentage,6311.317289,4.76,169240.47
2,Month 1,C003,Revenue percentage,3530.021385,2.66,94658.92
3,Month 1,C004,Revenue percentage,5744.316145,4.34,154036.11
4,Month 1,C005,Revenue percentage,16464.452593,12.43,441500.81
5,Month 1,C006,Revenue percentage,3384.693677,2.55,90761.90
6,Month 1,C007,Revenue percentage,3183.881424,2.40,85377.04
7,Month 1,C008,Revenue percentage,3674.910167,2.77,98544.17
8,Month 1,C009,Revenue percentage,3915.559982,2.96,104997.29
9,Month 1,C010,Revenue percentage,3190.000297,2.41,85541.12


In [106]:
managment_current_profitable['Management_Profit'] = (
    managment_current_profitable['Total_Revenue']
    - managment_current_profitable['Management Allocated Cost']
)
management_profit_rank = managment_current_profitable.sort_values(
    by='Management_Profit',
    ascending=False
)
management_profit_rank[['Customer Id','Management Allocated Cost','Total_Revenue','Management_Profit']]

,Customer Id,Management Allocated Cost,Total_Revenue,Management_Profit
4,C005,16464.452593,441500.81,425036.357407
0,C001,7442.366050,199569.99,192127.623950
1,C002,6311.317289,169240.47,162929.152711
3,C004,5744.316145,154036.11,148291.793855
11,C012,4476.272202,120033.01,115556.737798
12,C013,4322.187583,115901.17,111578.982417
29,C030,4275.662072,114653.57,110377.907928
15,C016,4229.090691,113404.74,109175.649309
21,C022,4107.564608,110145.97,106038.405392
13,C014,3982.230264,106785.08,102802.849736


Under the current management allocation method (revenue-based cost distribution), all customers appear profitable, indicating that the costing model does not differentiate operational efficiency or true cost-to-serve differences across customers.

### DOES EACH ACTIVITY CREATE VALUE?

In [107]:
m1Labour_df

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost,Data Quality Note
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38,NaN
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77,NaN
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47,NaN
3,Month 1,2026-01-01,Returns,629,89.11,4232.73,NaN
4,Month 1,2026-01-01,Rework,467,97.29,4621.28,NaN
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.00,NaN,NaN
6238,Month 1,2026-01-31,C030,Pick,688.00,NaN,NaN
6239,Month 1,2026-01-31,C030,Returns,27.00,NaN,NaN
6240,Month 1,2026-01-31,C030,Rework,4.00,NaN,NaN


In [108]:
m1Labour_df.iloc[:218,:]

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost,Data Quality Note
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38,NaN
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77,NaN
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47,NaN
3,Month 1,2026-01-01,Returns,629,89.11,4232.73,NaN
4,Month 1,2026-01-01,Rework,467,97.29,4621.28,NaN
...,...,...,...,...,...,...,...
213,Month 1,2026-01-31,Returns,568,80.47,3822.32,NaN
214,Month 1,2026-01-31,Rework,448,93.33,4433.18,NaN
215,Month 1,2026-01-31,Storage,8529,5.69,270.28,NaN
216,Month 1,2026-01-31,Urgent Order,279,34.88,1656.8,NaN


In [109]:
m1Revenue_df

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312,4.43,1382.16
6238,Month 1,2026-01-31,C030,Pick,688,0.14,96.32
6239,Month 1,2026-01-31,C030,Returns,27,8.54,230.58
6240,Month 1,2026-01-31,C030,Rework,4,11.43,45.72


In [110]:
m1Labour_df.drop('Data Quality Note',axis=1,inplace=True)
m1Labour_df

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47
3,Month 1,2026-01-01,Returns,629,89.11,4232.73
4,Month 1,2026-01-01,Rework,467,97.29,4621.28
...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.00,NaN
6238,Month 1,2026-01-31,C030,Pick,688.00,NaN
6239,Month 1,2026-01-31,C030,Returns,27.00,NaN
6240,Month 1,2026-01-31,C030,Rework,4.00,NaN


In [111]:
m1Labour_df.head()

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47
3,Month 1,2026-01-01,Returns,629,89.11,4232.73
4,Month 1,2026-01-01,Rework,467,97.29,4621.28


In [112]:
m1Labour_df.isnull().sum()

Month                 0
Date                  0
Activity Type         0
Units Processed       0
Labour Hours          2
Labour Cost        6024
dtype: int64

In [113]:
activity_labor_df = m1Labour_df.iloc[:217].copy()
activity_labor_df

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47
3,Month 1,2026-01-01,Returns,629,89.11,4232.73
4,Month 1,2026-01-01,Rework,467,97.29,4621.28
...,...,...,...,...,...,...
212,Month 1,2026-01-31,Receipt,1715,80.03,3801.43
213,Month 1,2026-01-31,Returns,568,80.47,3822.32
214,Month 1,2026-01-31,Rework,448,93.33,4433.18
215,Month 1,2026-01-31,Storage,8529,5.69,270.28


In [114]:
activity_labor_df.isnull().sum()

Month              0
Date               0
Activity Type      0
Units Processed    0
Labour Hours       0
Labour Cost        2
dtype: int64

In [115]:
activity_labor_df[activity_labor_df['Labour Cost'].isnull()]

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost
57,Month 1,2026-01-09,Pick,31350,198.55,NaN
64,Month 1,2026-01-10,Pick,32657,206.83,NaN


In [116]:
activity_labor_df['hourly_rate'] = (
    activity_labor_df['Labour Cost'] / activity_labor_df['Labour Hours']
)
activity_labor_df['hourly_rate'].describe()

count     215.0
unique    113.0
top        47.5
freq       70.0
Name: hourly_rate, dtype: float64

In [117]:
activity_labor_df['hourly_rate'].unique()

array([47.50002048760499, 47.49997562045931, 47.49993781867927,
       47.50005611042531, 47.50005139274334, 47.49913644214162,
       47.50000000000001, 47.500022838350155, 47.50002169856356,
       47.50005400151204, 47.5, 47.50006153846154, 47.49999999999999,
       47.5000547465236, 47.49994485496856, 47.49992104847624,
       47.500857632933105, 47.49997690851152, 47.4999772861491,
       47.50006605892457, 47.49994166374986, 47.49994132144115,
       47.499072356215216, 47.49997895179962, 47.50002255808707,
       47.50005681172593, 47.50005896921807, 47.49921507064364,
       47.49997352956748, 47.50006215040398, 47.50007606876616,
       47.500170940170946, 47.49997411070264, 47.500067999456,
       47.50002786912658, 47.50082644628099, 47.4999794770759, nan,
       47.50078988941548, 47.49997922639079, 47.499940341248056,
       47.49997631342082, 47.499977547263015, 47.50005480653294,
       47.500050347397035, 47.500059403587976, 47.499975651327006,
       47.500055230310394

In [118]:
activity_labor_df

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost,hourly_rate
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38,47.50002
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77,47.499976
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47,47.499938
3,Month 1,2026-01-01,Returns,629,89.11,4232.73,47.500056
4,Month 1,2026-01-01,Rework,467,97.29,4621.28,47.500051
...,...,...,...,...,...,...,...
212,Month 1,2026-01-31,Receipt,1715,80.03,3801.43,47.500062
213,Month 1,2026-01-31,Returns,568,80.47,3822.32,47.499938
214,Month 1,2026-01-31,Rework,448,93.33,4433.18,47.500054
215,Month 1,2026-01-31,Storage,8529,5.69,270.28,47.500879


In [119]:
activity_labor_df['hourly_rate']=activity_labor_df['hourly_rate'].round(2)
activity_labor_df['hourly_rate'].unique()

array([47.5, nan], dtype=object)

In [120]:
activity_labor_df[activity_labor_df['hourly_rate'].isnull()]

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost,hourly_rate
57,Month 1,2026-01-09,Pick,31350,198.55,NaN,NaN
64,Month 1,2026-01-10,Pick,32657,206.83,NaN,NaN


In [121]:
activity_labor_df['hourly_rate'] = activity_labor_df['hourly_rate'].fillna(47.5)

activity_labor_df.loc[
    activity_labor_df['Labour Cost'].isna(),
    'Labour Cost'
] = (
    activity_labor_df['Labour Hours'] *
    activity_labor_df['hourly_rate']
)
activity_labor_df[['Labour Hours','Labour Cost','hourly_rate']].isna().sum()

Labour Hours    0
Labour Cost     0
hourly_rate     0
dtype: int64

In [122]:
activity_labor_df

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost,hourly_rate
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38,47.5
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77,47.5
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47,47.5
3,Month 1,2026-01-01,Returns,629,89.11,4232.73,47.5
4,Month 1,2026-01-01,Rework,467,97.29,4621.28,47.5
...,...,...,...,...,...,...,...
212,Month 1,2026-01-31,Receipt,1715,80.03,3801.43,47.5
213,Month 1,2026-01-31,Returns,568,80.47,3822.32,47.5
214,Month 1,2026-01-31,Rework,448,93.33,4433.18,47.5
215,Month 1,2026-01-31,Storage,8529,5.69,270.28,47.5


In [123]:
##### finding aggregation in revenue

revenue_agg = m1Revenue_df.groupby(
    ['Date', 'Charge Type']
)['Charged Quantity'].sum().reset_index()
revenue_agg

,Date,Charge Type,Charged Quantity
0,2026-01-01,Dispatch,4513
1,2026-01-01,Pick,30609
2,2026-01-01,Receipt,1718
3,2026-01-01,Returns,600
4,2026-01-01,Rework,451
...,...,...,...
212,2026-01-31,Receipt,1715
213,2026-01-31,Returns,558
214,2026-01-31,Rework,431
215,2026-01-31,Storage,8529


In [124]:
revenue_agg = revenue_agg.rename(columns={
    'Charge Type': 'Activity Type'
})
revenue_agg

,Date,Activity Type,Charged Quantity
0,2026-01-01,Dispatch,4513
1,2026-01-01,Pick,30609
2,2026-01-01,Receipt,1718
3,2026-01-01,Returns,600
4,2026-01-01,Rework,451
...,...,...,...
212,2026-01-31,Receipt,1715
213,2026-01-31,Returns,558
214,2026-01-31,Rework,431
215,2026-01-31,Storage,8529


In [125]:
compare_with_revenue = activity_labor_df.groupby(
    ['Date', 'Activity Type']
)['Units Processed'].sum().reset_index().merge(
    revenue_agg,
    on=['Date', 'Activity Type'],
    how='inner'
)
compare_with_revenue

,Date,Activity Type,Units Processed,Charged Quantity
0,2026-01-01,Dispatch,4576,4513
1,2026-01-01,Pick,32383,30609
2,2026-01-01,Receipt,1723,1718
3,2026-01-01,Returns,629,600
4,2026-01-01,Rework,467,451
...,...,...,...,...
212,2026-01-31,Receipt,1715,1715
213,2026-01-31,Returns,568,558
214,2026-01-31,Rework,448,431
215,2026-01-31,Storage,8529,8529


In [126]:
compare_with_revenue['diff'] = compare_with_revenue['Units Processed'] - compare_with_revenue['Charged Quantity']
compare_with_revenue[compare_with_revenue['diff'] != 0]

,Date,Activity Type,Units Processed,Charged Quantity,diff
0,2026-01-01,Dispatch,4576,4513,63
1,2026-01-01,Pick,32383,30609,1774
2,2026-01-01,Receipt,1723,1718,5
3,2026-01-01,Returns,629,600,29
4,2026-01-01,Rework,467,451,16
...,...,...,...,...,...
210,2026-01-31,Dispatch,4609,4462,147
211,2026-01-31,Pick,31256,30067,1189
213,2026-01-31,Returns,568,558,10
214,2026-01-31,Rework,448,431,17


In [127]:
compare_with_revenue.groupby('Activity Type')['diff'].sum()

Activity Type
Dispatch         3884
Pick            31935
Receipt           108
Returns           674
Rework            466
Storage           452
Urgent Order      326
Name: diff, dtype: object

In [128]:
comparing_reveune_quantity=compare_with_revenue.groupby('Activity Type')['diff'].sum()
comparing_reveune_quantity

Activity Type
Dispatch         3884
Pick            31935
Receipt           108
Returns           674
Rework            466
Storage           452
Urgent Order      326
Name: diff, dtype: object

In [129]:
comparing_revenue_quantity = compare_with_revenue.groupby(
    'Activity Type'
)['diff'].sum().reset_index()
comparing_revenue_quantity = comparing_revenue_quantity.sort_values(
    by='diff',
    ascending=False
)
comparing_revenue_quantity

,Activity Type,diff
1,Pick,31935
0,Dispatch,3884
3,Returns,674
4,Rework,466
5,Storage,452
6,Urgent Order,326
2,Receipt,108


In [130]:
comparing_351_activity_quantity = volume_compare[['Charge Type', 'diff']].copy()

comparing_351_activity_quantity = comparing_351_activity_quantity.sort_values(
    by='diff',
    ascending=False
)

comparing_351_activity_quantity

,Charge Type,diff
0,Pick,31935.0
2,Dispatch,3884.0
4,Returns,674.0
5,Rework,466.0
1,Storage,452.0
6,Urgent Order,326.0
3,Receipt,108.0


In [131]:
m1Activities_df

,Month,Date,Customer Id,Activity Type,Quantity
0,Month 1,2026-01-01,C001,Storage,913.0
1,Month 1,2026-01-01,C001,Receipt,33.0
2,Month 1,2026-01-01,C001,Dispatch,54.0
3,Month 1,2026-01-01,C001,Pick,231.0
4,Month 1,2026-01-01,C001,Returns,3.0
...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0
6238,Month 1,2026-01-31,C030,Pick,688.0
6239,Month 1,2026-01-31,C030,Returns,27.0
6240,Month 1,2026-01-31,C030,Rework,4.0


In [132]:
##### finding aggregation in revenue

activity_agg = m1Activities_df.groupby(
    ['Date', 'Activity Type']
)['Quantity'].sum().reset_index()


activity_agg

,Date,Activity Type,Quantity
0,2026-01-01,Dispatch,4576.0
1,2026-01-01,Pick,32383.0
2,2026-01-01,Receipt,1723.0
3,2026-01-01,Returns,629.0
4,2026-01-01,Rework,467.0
...,...,...,...
212,2026-01-31,Receipt,1715.0
213,2026-01-31,Returns,568.0
214,2026-01-31,Rework,448.0
215,2026-01-31,Storage,8529.0


In [133]:
compare_with_activity = activity_labor_df.groupby(
    ['Date', 'Activity Type']
)['Units Processed'].sum().reset_index().merge(
    activity_agg,
    on=['Date', 'Activity Type'],
    how='inner'
)
compare_with_activity

,Date,Activity Type,Units Processed,Quantity
0,2026-01-01,Dispatch,4576,4576.0
1,2026-01-01,Pick,32383,32383.0
2,2026-01-01,Receipt,1723,1723.0
3,2026-01-01,Returns,629,629.0
4,2026-01-01,Rework,467,467.0
...,...,...,...,...
212,2026-01-31,Receipt,1715,1715.0
213,2026-01-31,Returns,568,568.0
214,2026-01-31,Rework,448,448.0
215,2026-01-31,Storage,8529,8529.0


In [134]:
compare_with_activity['diff'] = compare_with_activity['Units Processed'] - compare_with_activity['Quantity']
compare_with_activity[compare_with_activity['diff'] != 0]

,Date,Activity Type,Units Processed,Quantity,diff


### WHICH ACTIVITY CREATE OR DESTROY VALUE ??

##### Activity Revenue (monthly)

In [135]:
activity_revenue = m1Revenue_df.groupby(
    'Charge Type'
).agg(
    Revenue=('Revenue', 'sum'),
    Quantity_Units_Processed=('Charged Quantity', 'sum')
).reset_index().rename(columns={'Charge Type': 'Activity Type'})
activity_revenue

,Activity Type,Revenue,Quantity_Units_Processed
0,Dispatch,508004.18,132890
1,Pick,137829.55,961653
2,Receipt,196185.27,58237
3,Returns,159046.73,16417
4,Rework,130880.04,11198
5,Storage,2289819.88,271319
6,Urgent Order,131274.27,8877


In [136]:
activity_cost = activity_labor_df.groupby(
    'Activity Type'
).agg(
    Cost=('Labour Cost', 'sum'),
    Units_Processed=('Units Processed', 'sum')
).reset_index()
activity_cost

,Activity Type,Cost,Units_Processed
0,Dispatch,346494.45,136774
1,Pick,298904.18,993588
2,Receipt,129331.12,58345
3,Returns,115009.36,17091
4,Rework,115425.49,11664
5,Storage,8606.53,271771
6,Urgent Order,54643.04,9203


In [137]:
activity_value = activity_revenue.merge(
    activity_cost,
    on='Activity Type',
    how='inner'
)
activity_value

,Activity Type,Revenue,Quantity_Units_Processed,Cost,Units_Processed
0,Dispatch,508004.18,132890,346494.45,136774
1,Pick,137829.55,961653,298904.18,993588
2,Receipt,196185.27,58237,129331.12,58345
3,Returns,159046.73,16417,115009.36,17091
4,Rework,130880.04,11198,115425.49,11664
5,Storage,2289819.88,271319,8606.53,271771
6,Urgent Order,131274.27,8877,54643.04,9203


In [138]:
activity_value['Profit'] = activity_value['Revenue'] - activity_value['Cost']
activity_value

,Activity Type,Revenue,Quantity_Units_Processed,Cost,Units_Processed,Profit
0,Dispatch,508004.18,132890,346494.45,136774,161509.73
1,Pick,137829.55,961653,298904.18,993588,-161074.63
2,Receipt,196185.27,58237,129331.12,58345,66854.15
3,Returns,159046.73,16417,115009.36,17091,44037.37
4,Rework,130880.04,11198,115425.49,11664,15454.55
5,Storage,2289819.88,271319,8606.53,271771,2281213.35
6,Urgent Order,131274.27,8877,54643.04,9203,76631.23


In [139]:
activity_value[['Activity Type','Profit']]

,Activity Type,Profit
0,Dispatch,161509.73
1,Pick,-161074.63
2,Receipt,66854.15
3,Returns,44037.37
4,Rework,15454.55
5,Storage,2281213.35
6,Urgent Order,76631.23


With Billing , it shows pick activity destroy value but we gonna compare with true operational volumne

In [140]:
m1Revenue_df

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312,4.43,1382.16
6238,Month 1,2026-01-31,C030,Pick,688,0.14,96.32
6239,Month 1,2026-01-31,C030,Returns,27,8.54,230.58
6240,Month 1,2026-01-31,C030,Rework,4,11.43,45.72


In [141]:
customer_pricing_df

,Customer Id,Activity Type,Unit Type,Contract Rate
0,C001,Storage,pallet_day,8.96
1,C001,Receipt,pallet,3.92
2,C001,Dispatch,order,4.48
3,C001,Pick,pick,0.17
4,C001,Returns,return,11.20
...,...,...,...,...
205,C030,Dispatch,order,4.43
206,C030,Pick,pick,0.14
207,C030,Returns,return,8.54
208,C030,Rework,job,11.43


In [142]:
true_revenue_from_activity = m1Activities_df.merge(
    customer_pricing_df,
    on=['Customer Id', 'Activity Type'],
    how='left'
)
true_revenue_from_activity.head(10)

,Month,Date,Customer Id,Activity Type,Quantity,Unit Type,Contract Rate
0,Month 1,2026-01-01,C001,Storage,913.0,pallet_day,8.96
1,Month 1,2026-01-01,C001,Receipt,33.0,pallet,3.92
2,Month 1,2026-01-01,C001,Dispatch,54.0,order,4.48
3,Month 1,2026-01-01,C001,Pick,231.0,pick,0.17
4,Month 1,2026-01-01,C001,Returns,3.0,return,11.20
5,Month 1,2026-01-01,C002,Storage,244.0,pallet_day,6.24
6,Month 1,2026-01-01,C002,Receipt,88.0,pallet,2.73
7,Month 1,2026-01-01,C002,Dispatch,343.0,order,3.12
8,Month 1,2026-01-01,C002,Pick,5346.0,pick,0.12
9,Month 1,2026-01-01,C002,Returns,25.0,return,7.80


In [143]:
true_revenue_from_activity['True_Revenue'] = (
    true_revenue_from_activity['Quantity'] *
    true_revenue_from_activity['Contract Rate']
)
true_revenue_from_activity.head()

,Month,Date,Customer Id,Activity Type,Quantity,Unit Type,Contract Rate,True_Revenue
0,Month 1,2026-01-01,C001,Storage,913.0,pallet_day,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33.0,pallet,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54.0,order,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231.0,pick,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3.0,return,11.20,33.60


In [144]:
true_revenue_from_activity[['Activity Type', 'Quantity', 'Contract Rate', 'True_Revenue']].head()

,Activity Type,Quantity,Contract Rate,True_Revenue
0,Storage,913.0,8.96,8180.48
1,Receipt,33.0,3.92,129.36
2,Dispatch,54.0,4.48,241.92
3,Pick,231.0,0.17,39.27
4,Returns,3.0,11.20,33.60


In [145]:
activity_revenue_true = true_revenue_from_activity.groupby(
    'Activity Type'
)[['True_Revenue','Quantity']].sum().reset_index()
activity_revenue_true

,Activity Type,True_Revenue,Quantity
0,Dispatch,524233.02,136774.0
1,Pick,142948.00,993588.0
2,Receipt,196552.69,58345.0
3,Returns,164834.40,17091.0
4,Rework,135953.03,11664.0
5,Storage,2293527.44,271771.0
6,Urgent Order,135496.05,9203.0


In [146]:
activity_cost = activity_labor_df.groupby(
    'Activity Type'
)[['Labour Cost','Units Processed']].sum().reset_index()
activity_cost

,Activity Type,Labour Cost,Units Processed
0,Dispatch,346494.45,136774
1,Pick,298904.18,993588
2,Receipt,129331.12,58345
3,Returns,115009.36,17091
4,Rework,115425.49,11664
5,Storage,8606.53,271771
6,Urgent Order,54643.04,9203


In [147]:
activity_profit = activity_revenue_true.merge(
    activity_cost,
    on='Activity Type',
    how='inner'
)
activity_profit['Profit'] = (
    activity_profit['True_Revenue'] - activity_profit['Labour Cost']
)
activity_profit.sort_values('Profit', ascending=False)

,Activity Type,True_Revenue,Quantity,Labour Cost,Units Processed,Profit
5,Storage,2293527.44,271771.0,8606.53,271771,2284920.91
0,Dispatch,524233.02,136774.0,346494.45,136774,177738.57
6,Urgent Order,135496.05,9203.0,54643.04,9203,80853.01
2,Receipt,196552.69,58345.0,129331.12,58345,67221.57
3,Returns,164834.40,17091.0,115009.36,17091,49825.04
4,Rework,135953.03,11664.0,115425.49,11664,20527.54
1,Pick,142948.00,993588.0,298904.18,993588,-155956.18


In [148]:
activity_profit['Cost_per_Unit'] = (
    activity_profit['Labour Cost'] / activity_profit['Units Processed']
)
activity_profit

,Activity Type,True_Revenue,Quantity,Labour Cost,Units Processed,Profit,Cost_per_Unit
0,Dispatch,524233.02,136774.0,346494.45,136774,177738.57,2.533336
1,Pick,142948.00,993588.0,298904.18,993588,-155956.18,0.300833
2,Receipt,196552.69,58345.0,129331.12,58345,67221.57,2.216662
3,Returns,164834.40,17091.0,115009.36,17091,49825.04,6.729235
4,Rework,135953.03,11664.0,115425.49,11664,20527.54,9.895875
5,Storage,2293527.44,271771.0,8606.53,271771,2284920.91,0.031668
6,Urgent Order,135496.05,9203.0,54643.04,9203,80853.01,5.937525


In [149]:
(activity_profit['Quantity']==activity_profit['Units Processed']).all()

np.True_

In [150]:
activity_profit['Revenue per unit'] = (
    activity_profit['True_Revenue'] / activity_profit['Units Processed']
)
activity_profit

,Activity Type,True_Revenue,Quantity,Labour Cost,Units Processed,Profit,Cost_per_Unit,Revenue per unit
0,Dispatch,524233.02,136774.0,346494.45,136774,177738.57,2.533336,3.832841
1,Pick,142948.00,993588.0,298904.18,993588,-155956.18,0.300833,0.14387
2,Receipt,196552.69,58345.0,129331.12,58345,67221.57,2.216662,3.368801
3,Returns,164834.40,17091.0,115009.36,17091,49825.04,6.729235,9.644515
4,Rework,135953.03,11664.0,115425.49,11664,20527.54,9.895875,11.655781
5,Storage,2293527.44,271771.0,8606.53,271771,2284920.91,0.031668,8.439191
6,Urgent Order,135496.05,9203.0,54643.04,9203,80853.01,5.937525,14.723031


In [151]:
activity_profit[['Activity Type','True_Revenue','Labour Cost','Units Processed','Profit','Cost_per_Unit','Revenue per unit']]

,Activity Type,True_Revenue,Labour Cost,Units Processed,Profit,Cost_per_Unit,Revenue per unit
0,Dispatch,524233.02,346494.45,136774,177738.57,2.533336,3.832841
1,Pick,142948.00,298904.18,993588,-155956.18,0.300833,0.14387
2,Receipt,196552.69,129331.12,58345,67221.57,2.216662,3.368801
3,Returns,164834.40,115009.36,17091,49825.04,6.729235,9.644515
4,Rework,135953.03,115425.49,11664,20527.54,9.895875,11.655781
5,Storage,2293527.44,8606.53,271771,2284920.91,0.031668,8.439191
6,Urgent Order,135496.05,54643.04,9203,80853.01,5.937525,14.723031


Activity Type: STORAGE
- Lowest cost per unit
- Very high revenue contribution
- Highest total profit contribution
Conclusion: Storage is the core profit driver of the business. It is highly efficient and should be scaled confidently.


Activity Type: DISPATCH
- Moderate cost per unit
- Strong revenue contribution
- Positive overall profit
Conclusion: Dispatch is a stable and necessary operational activity. It supports revenue generation effectively and should be maintained with efficiency improvements only.


Activity Type: RECEIPT
- Low to moderate cost per unit
- Moderate revenue contribution
- Positive profit
Conclusion: Receipt is an efficient backend process that contributes steadily to profitability. It is not a concern area.


Activity Type: URGENT ORDER
- High cost per unit
- High revenue per unit
- Strong positive profit
Conclusion: Urgent Orders are a premium-priced service. They are profitable due to pricing, but operationally expensive to fulfill.


Activity Type: RETURNS
- High cost per unit
- Moderate revenue contribution
- Lower efficiency compared to other activities
Conclusion: Returns create operational cost pressure. This is likely driven by customer behavior or service quality issues and should be reduced.


Activity Type: REWORK
- Very high cost per unit
- Low revenue contribution
- Low total profit
Conclusion: Rework is a clear inefficiency indicator. It reflects process failures and should be minimized as a priority.


Activity Type: PICK
- Highest operational volume in the business
- Very low revenue per unit
- Low cost per unit but extremely large total volume
- Results in negative profitability
Conclusion: Pick is structurally underpriced. It is the main operational driver but creates a loss due to insufficient pricing rather than inefficiency.

In [152]:
import plotly.express as px
import pandas as pd

# Data with your categorization
data = {
    'Activity Type': ['Storage', 'Dispatch', 'Receipt', 'Urgent Order', 'Returns', 'Rework', 'Pick'],
    'Total_Profit': [2284920.91, 177738.57, 67221.57, 80853.01, 49825.04, 20527.54, -155956.18],
    'Cost_per_Unit': [0.03, 2.53, 2.21, 5.93, 6.72, 9.89, 0.30],
    'Total_Revenue': [2293527.44, 524233.02, 196552.69, 135496.05, 164834.40, 135953.03, 142948.00],
    'Category': [
        'Core Value Creator', 
        'Value Creator', 
        'Stable Contributor', 
        'Premium but Costly', 
        'Value Eroder', 
        'Value Destroyer', 
        'Major Value Destroyer'
    ],
    'Color': ['#00C853', '#00C853', '#00C853', '#FFC107', '#FF9800', '#F44336', '#D32F2F']
}

df = pd.DataFrame(data)

# Create the scatter plot (Value Matrix)
fig = px.scatter(
    df,
    x='Total_Profit',
    y='Cost_per_Unit',
    size='Total_Revenue',
    color='Category',
    color_discrete_map={
        'Core Value Creator': '#00C853',
        'Value Creator': '#00C853',
        'Stable Contributor': '#00C853',
        'Premium but Costly': '#FFC107',
        'Value Eroder': '#FF9800',
        'Value Destroyer': '#F44336',
        'Major Value Destroyer': '#D32F2F'
    },
    text='Activity Type',
    title="<b>Activities Value Matrix - Month 1</b><br><sup>X = Total Profit | Y = Cost per Unit (Lower is Better) | Size = Total Revenue</sup>",
    labels={
        'Total_Profit': 'Total Profit ($)',
        'Cost_per_Unit': 'Cost per Unit ($)'
    },
    size_max=70
)

fig.update_traces(
    textposition='top center',
    textfont=dict(size=11, color='white'),
    marker=dict(
        line=dict(width=1.5, color='white'),
        opacity=0.9
    ),
    hovertemplate="<b>%{text}</b><br>" +
                  "Category: %{fullData.name}<br>" +
                  "Total Profit: $%{x:,.0f}<br>" +
                  "Cost per Unit: $%{y:.2f}<br>" +
                  "Total Revenue: $%{marker.size:,.0f}<extra></extra>"
)

fig.update_layout(
    height=700,
    width=1000,
    title_font=dict(size=18, color='white'),
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(
        showgrid=True,
        gridcolor='#444444',
        zeroline=True,
        zerolinecolor='#888888',
        title_font=dict(color='white')
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='#444444',
        zeroline=True,
        zerolinecolor='#888888',
        title_font=dict(color='white'),
        autorange='reversed'   # Lower cost per unit = better (higher on chart)
    ),
    legend=dict(
        title="Category",
        orientation="v",
        yanchor="middle",
        y=0.5,
        font=dict(size=11)
    )
)

fig.show()

In [153]:
import plotly.express as px

fig = px.bar(
    df.sort_values('Total_Profit', ascending=True),
    x='Total_Profit',
    y='Activity Type',
    orientation='h',
    title="<b>Total Profit by Activity Type</b>",
    text='Total_Profit',
    color='Total_Profit',
    color_continuous_scale=['#D32F2F', '#FFC107', '#00C853']
)

fig.update_traces(
    texttemplate='$%{text:,.0f}',
    textposition='outside',
    textfont=dict(size=11, color='white')
)

fig.update_layout(
    height=500,
    width=850,
    title_font=dict(size=18, color='white'),
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(showgrid=True, gridcolor='#444444'),
    yaxis_title="Activity Type",
    xaxis_title="Total Profit ($)",
    coloraxis_colorbar=dict(title="Profit ($)")
)

fig.show()

In [154]:
import plotly.express as px

fig = px.bar(
    df.sort_values('Cost_per_Unit', ascending=False),
    x='Cost_per_Unit',
    y='Activity Type',
    orientation='h',
    title="<b>Cost per Unit by Activity Type</b>",
    text='Cost_per_Unit',
    color='Cost_per_Unit',
    color_continuous_scale='Reds'
)

fig.update_traces(
    texttemplate='$%{text:.2f}',
    textposition='outside',
    textfont=dict(size=11, color='white')
)

fig.update_layout(
    height=500,
    width=850,
    title_font=dict(size=18, color='white'),
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(showgrid=True, gridcolor='#444444'),
    yaxis_title="Activity Type",
    xaxis_title="Cost per Unit ($)",
    coloraxis_colorbar=dict(title="Cost per Unit ($)")
)

fig.show()

### EXCEPTIONS : ANY UNEXPECTED ACTIVITIES

In [155]:
m1Exceptions_df

,Month,Date,Customer,Exception Type,Count
0,Month 1,2026-01-01,C003,Possible Unrecovered Activity,29
1,Month 1,2026-01-01,C003,Possible Unrecovered Activity,1774
2,Month 1,2026-01-01,C003,Possible Unrecovered Activity,4
3,Month 1,2026-01-01,C004,Returns,8
4,Month 1,2026-01-01,C004,Rework,3
...,...,...,...,...,...
169,Month 1,2026-01-31,C003,Possible Unrecovered Activity,143
170,Month 1,2026-01-31,C003,Possible Unrecovered Activity,1139
171,Month 1,2026-01-31,C004,Returns,8
172,Month 1,2026-01-31,C004,Rework,17


In [156]:
m1Exceptions_df['Customer'].unique()

<ArrowStringArray>
['C003', 'C004']
Length: 2, dtype: str

In [157]:
m1Exceptions_df['Exception Type'].unique()

<ArrowStringArray>
['Possible Unrecovered Activity', 'Returns', 'Rework', 'Urgent Order']
Length: 4, dtype: str

In [158]:
exception_counts = (
    m1Exceptions_df[
        m1Exceptions_df['Customer'].isin(['C003', 'C004'])
    ]
    .groupby(['Customer', 'Exception Type'])['Count']
    .sum()
    .reset_index()
)
exception_counts

,Customer,Exception Type,Count
0,C003,Possible Unrecovered Activity,35991
1,C004,Returns,351
2,C004,Rework,202
3,C004,Urgent Order,178


### INVENTORY :WHAT DOES WAREHOUSE HAS IN ITS INVENTORY

In [159]:
m1Inventory_df

,Month,Date,Customer Id,Units On Hand,Avg Days In Stock,Inventory Class
0,Month 1,2026-01-01,C001,4886,30,A
1,Month 1,2026-01-01,C002,2812,5,A
2,Month 1,2026-01-01,C003,2066,53,A
3,Month 1,2026-01-01,C004,4476,25,B
4,Month 1,2026-01-01,C005,3061,84,C
...,...,...,...,...,...,...
925,Month 1,2026-01-31,C026,4876,65,A
926,Month 1,2026-01-31,C027,4935,86,C
927,Month 1,2026-01-31,C028,3864,25,A
928,Month 1,2026-01-31,C029,1930,64,B


##### we will do this during customer level

### IS EACH CUSTOMER PROFITABLE?

In [160]:
m1Revenue_df

,Month,Date,Customer Id,Charge Type,Charged Quantity,Rate,Revenue
0,Month 1,2026-01-01,C001,Storage,913,8.96,8180.48
1,Month 1,2026-01-01,C001,Receipt,33,3.92,129.36
2,Month 1,2026-01-01,C001,Dispatch,54,4.48,241.92
3,Month 1,2026-01-01,C001,Pick,231,0.17,39.27
4,Month 1,2026-01-01,C001,Returns,3,11.20,33.60
...,...,...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312,4.43,1382.16
6238,Month 1,2026-01-31,C030,Pick,688,0.14,96.32
6239,Month 1,2026-01-31,C030,Returns,27,8.54,230.58
6240,Month 1,2026-01-31,C030,Rework,4,11.43,45.72


Although we have calculated revenue per customer, this revenue figure is not fully reliable because some activity quantities were not properly recorded or charged.
We have already analyzed the activities and mathematically identified which ones create value and which ones destroy value.
To accurately determine which customers are profitable, we now need to calculate the true Cost to Serve for each customer. Only then can we calculate real profit per customer (Revenue – Cost to Serve).
Additionally, since we know which activities destroy value, customers who heavily use those activities (such as Pick, Rework, or Returns) are likely to show lower profitability, even if their revenue appears decent.

In [161]:
profit_view.head()

,Rank,Customer Id,Total_Revenue,Management Allocated Cost,Estimated_Profit,Profit_Margin_%
0,1,C005,441500.81,16464.452593,425036.357407,96.27
1,2,C001,199569.99,7442.366050,192127.623950,96.27
2,3,C002,169240.47,6311.317289,162929.152711,96.27
3,4,C004,154036.11,5744.316145,148291.793855,96.27
4,5,C012,120033.01,4476.272202,115556.737798,96.27


Current Management Profit Calculation Method (Problem):
Management is currently calculating customer profit using this simple formula:
Profit = Total Revenue – Management Allocated Cost
Because management allocates cost based on revenue percentage, the result is that almost every customer shows nearly the same profit margin (~96.27%).
This creates a false picture:

High-revenue customers appear highly profitable.
Low-revenue customers also appear equally profitable.
As a result, management sees revenue growing and assumes profitability is also growing proportionally.

This is one of the main reasons why management cannot clearly see the real profitability problem in the business.

Key Issues with Current Method:

It makes all customers look equally profitable (flat margins).
It hides which customers are actually good or bad for the business.
It does not reflect the true cost to serve each customer.
It ignores the fact that some activities (like Storage & Dispatch) create value while others (like Pick, Rework, Returns) destroy value.

What We Need to Do Next:
To give management a clearer picture, we need to move from this flawed method to calculating real customer profitability by finding the true Cost to Serve per customer. This will help identify:

Which customers are genuinely profitable.
Which customers are quietly destroying profit.
How different activities impact customer-level profitability.

In [162]:
customer_based_labour_df=m1Labour_df.iloc[217:,:].drop('Labour Cost',axis=1)
customer_based_labour_df

,Month,Date,Activity Type,Units Processed,Labour Hours
217,Month 1,2026-01-02,C003,Receipt,48.0
218,Month 1,2026-01-02,C003,Dispatch,178.0
219,Month 1,2026-01-02,C003,Pick,3636.0
220,Month 1,2026-01-02,C003,Returns,16.0
221,Month 1,2026-01-02,C003,Rework,3.0
...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0
6238,Month 1,2026-01-31,C030,Pick,688.0
6239,Month 1,2026-01-31,C030,Returns,27.0
6240,Month 1,2026-01-31,C030,Rework,4.0


In [163]:
customer_based_df = (
    m1Labour_df.iloc[217:]
      .rename(columns={
          'Activity Type':'Customer Id',
          'Units Processed':'Activity Type',
          'Labour Hours':'Quantity'
      }).drop('Labour Cost',axis=1)
)
customer_based_df

,Month,Date,Customer Id,Activity Type,Quantity
217,Month 1,2026-01-02,C003,Receipt,48.0
218,Month 1,2026-01-02,C003,Dispatch,178.0
219,Month 1,2026-01-02,C003,Pick,3636.0
220,Month 1,2026-01-02,C003,Returns,16.0
221,Month 1,2026-01-02,C003,Rework,3.0
...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0
6238,Month 1,2026-01-31,C030,Pick,688.0
6239,Month 1,2026-01-31,C030,Returns,27.0
6240,Month 1,2026-01-31,C030,Rework,4.0


In [164]:
customer_based_df.isnull().sum()

Month            0
Date             0
Customer Id      0
Activity Type    0
Quantity         2
dtype: int64

In [165]:
customer_based_df[customer_based_df['Quantity'].isnull()]

,Month,Date,Customer Id,Activity Type,Quantity
1072,Month 1,2026-01-06,C011,Pick,NaN
1276,Month 1,2026-01-07,C011,Pick,NaN


2 data values are missing for c011, i.e labor hours system is flawed or not recorded

In [166]:
m1Activities_df

,Month,Date,Customer Id,Activity Type,Quantity
0,Month 1,2026-01-01,C001,Storage,913.0
1,Month 1,2026-01-01,C001,Receipt,33.0
2,Month 1,2026-01-01,C001,Dispatch,54.0
3,Month 1,2026-01-01,C001,Pick,231.0
4,Month 1,2026-01-01,C001,Returns,3.0
...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0
6238,Month 1,2026-01-31,C030,Pick,688.0
6239,Month 1,2026-01-31,C030,Returns,27.0
6240,Month 1,2026-01-31,C030,Rework,4.0


In [167]:
customer_based_df

,Month,Date,Customer Id,Activity Type,Quantity
217,Month 1,2026-01-02,C003,Receipt,48.0
218,Month 1,2026-01-02,C003,Dispatch,178.0
219,Month 1,2026-01-02,C003,Pick,3636.0
220,Month 1,2026-01-02,C003,Returns,16.0
221,Month 1,2026-01-02,C003,Rework,3.0
...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0
6238,Month 1,2026-01-31,C030,Pick,688.0
6239,Month 1,2026-01-31,C030,Returns,27.0
6240,Month 1,2026-01-31,C030,Rework,4.0


In [168]:
compare_customer = (
    customer_based_df
    .groupby(['Date','Customer Id','Activity Type'])['Quantity']
    .sum()
    .reset_index()
)

activity_customer = (
    m1Activities_df
    .groupby(['Date','Customer Id','Activity Type'])['Quantity']
    .sum()
    .reset_index()
)

compare = activity_customer.merge(
    compare_customer,
    on=['Date','Customer Id','Activity Type'],
    suffixes=('_activity','_customer')
)

compare['diff'] = (
    compare['Quantity_activity']
    - compare['Quantity_customer']
)

compare[compare['diff'] != 0]

,Date,Customer Id,Activity Type,Quantity_activity,Quantity_customer,diff


In [169]:
missing_rows = m1Activities_df.merge(
    customer_based_df,
    on=['Date','Customer Id','Activity Type'],
    how='left',
    indicator=True
)

missing_rows_df=missing_rows[missing_rows['_merge'] == 'left_only']
missing_rows_df

,Month_x,Date,Customer Id,Activity Type,Quantity_x,Month_y,Quantity_y,_merge
0,Month 1,2026-01-01,C001,Storage,913.0,NaN,NaN,left_only
1,Month 1,2026-01-01,C001,Receipt,33.0,NaN,NaN,left_only
2,Month 1,2026-01-01,C001,Dispatch,54.0,NaN,NaN,left_only
3,Month 1,2026-01-01,C001,Pick,231.0,NaN,NaN,left_only
4,Month 1,2026-01-01,C001,Returns,3.0,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...
212,Month 1,2026-01-02,C002,Pick,3357.0,NaN,NaN,left_only
213,Month 1,2026-01-02,C002,Returns,3.0,NaN,NaN,left_only
214,Month 1,2026-01-02,C002,Rework,19.0,NaN,NaN,left_only
215,Month 1,2026-01-02,C002,Urgent Order,4.0,NaN,NaN,left_only


In [170]:
missing_rows_df['Date'].unique()

<DatetimeArray>
['2026-01-01 00:00:00', '2026-01-02 00:00:00']
Length: 2, dtype: datetime64[us]

In [171]:
missing_rows_df['Customer Id'].unique()

<ArrowStringArray>
['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008', 'C009',
 'C010', 'C011', 'C012', 'C013', 'C014', 'C015', 'C016', 'C017', 'C018',
 'C019', 'C020', 'C021', 'C022', 'C023', 'C024', 'C025', 'C026', 'C027',
 'C028', 'C029', 'C030']
Length: 30, dtype: str

In [172]:
missing_rows_df['Activity Type'].unique()

array(['Storage', 'Receipt', 'Dispatch', 'Pick', 'Returns', 'Rework',
       'Urgent Order'], dtype=object)

The dataset has two parts: a reliable activity-level table with units processed and true labour cost, and a customer-level table showing only activity quantities. The customer table is not a cost source and cannot be used directly for profitability. True costs must come from the activity-level data. Customer profitability should be calculated by allocating these activity costs based on each customer’s usage.

In [173]:
activity_labor_df

,Month,Date,Activity Type,Units Processed,Labour Hours,Labour Cost,hourly_rate
0,Month 1,2026-01-01,Dispatch,4576,244.05,11592.38,47.5
1,Month 1,2026-01-01,Pick,32383,205.09,9741.77,47.5
2,Month 1,2026-01-01,Receipt,1723,80.41,3819.47,47.5
3,Month 1,2026-01-01,Returns,629,89.11,4232.73,47.5
4,Month 1,2026-01-01,Rework,467,97.29,4621.28,47.5
...,...,...,...,...,...,...,...
212,Month 1,2026-01-31,Receipt,1715,80.03,3801.43,47.5
213,Month 1,2026-01-31,Returns,568,80.47,3822.32,47.5
214,Month 1,2026-01-31,Rework,448,93.33,4433.18,47.5
215,Month 1,2026-01-31,Storage,8529,5.69,270.28,47.5


In [174]:
activity_cost_baseline = activity_labor_df.groupby('Activity Type').agg(
    Total_Units_Processed=('Units Processed', 'sum'),
    Total_Labour_Hours=('Labour Hours', 'sum'),
    Total_Labour_Cost=('Labour Cost', 'sum')
).reset_index()

activity_cost_baseline

,Activity Type,Total_Units_Processed,Total_Labour_Hours,Total_Labour_Cost
0,Dispatch,136774,7294.62,346494.45
1,Pick,993588,6292.72,298904.18
2,Receipt,58345,2722.76,129331.12
3,Returns,17091,2421.25,115009.36
4,Rework,11664,2430.01,115425.49
5,Storage,271771,181.19,8606.53
6,Urgent Order,9203,1150.38,54643.04


In [175]:
activity_cost_baseline['Cost_per_Unit'] = (
    activity_cost_baseline['Total_Labour_Cost'] /
    activity_cost_baseline['Total_Units_Processed']
)
activity_cost_baseline

,Activity Type,Total_Units_Processed,Total_Labour_Hours,Total_Labour_Cost,Cost_per_Unit
0,Dispatch,136774,7294.62,346494.45,2.533336
1,Pick,993588,6292.72,298904.18,0.300833
2,Receipt,58345,2722.76,129331.12,2.216662
3,Returns,17091,2421.25,115009.36,6.729235
4,Rework,11664,2430.01,115425.49,9.895875
5,Storage,271771,181.19,8606.53,0.031668
6,Urgent Order,9203,1150.38,54643.04,5.937525



Revenue	revenue_df	:What we charge
Cost	activity_labor_df	:What it costs us
Contract rate	pricing_df	:Pricing logic (revenue driver only)

In [176]:
m1Activities_df

,Month,Date,Customer Id,Activity Type,Quantity
0,Month 1,2026-01-01,C001,Storage,913.0
1,Month 1,2026-01-01,C001,Receipt,33.0
2,Month 1,2026-01-01,C001,Dispatch,54.0
3,Month 1,2026-01-01,C001,Pick,231.0
4,Month 1,2026-01-01,C001,Returns,3.0
...,...,...,...,...,...
6237,Month 1,2026-01-31,C030,Dispatch,312.0
6238,Month 1,2026-01-31,C030,Pick,688.0
6239,Month 1,2026-01-31,C030,Returns,27.0
6240,Month 1,2026-01-31,C030,Rework,4.0


In [177]:
customer_activity_df = m1Activities_df.copy()

customer_activity_df = customer_activity_df.groupby(
    ['Customer Id', 'Activity Type']
)['Quantity'].sum().reset_index()

customer_activity_df

,Customer Id,Activity Type,Quantity
0,C001,Dispatch,2639.0
1,C001,Pick,16790.0
2,C001,Receipt,1145.0
3,C001,Returns,72.0
4,C001,Rework,16.0
...,...,...,...
205,C030,Receipt,1849.0
206,C030,Returns,639.0
207,C030,Rework,365.0
208,C030,Storage,8051.0


In [178]:
customer_activity_df['Activity Type'].unique()

<ArrowStringArray>
['Dispatch', 'Pick', 'Receipt', 'Returns', 'Rework', 'Storage',
 'Urgent Order']
Length: 7, dtype: str

In [179]:
customer_activity_df['Customer Id'].nunique()

30

In [180]:
customer_activity_df.isnull().sum()

Customer Id      0
Activity Type    0
Quantity         0
dtype: int64

In [181]:
customer_cost_df = customer_activity_df.merge(
    activity_cost_baseline[['Activity Type', 'Cost_per_Unit']],
    on='Activity Type',
    how='left'
)

customer_cost_df

,Customer Id,Activity Type,Quantity,Cost_per_Unit
0,C001,Dispatch,2639.0,2.533336
1,C001,Pick,16790.0,0.300833
2,C001,Receipt,1145.0,2.216662
3,C001,Returns,72.0,6.729235
4,C001,Rework,16.0,9.895875
...,...,...,...,...
205,C030,Receipt,1849.0,2.216662
206,C030,Returns,639.0,6.729235
207,C030,Rework,365.0,9.895875
208,C030,Storage,8051.0,0.031668


In [182]:
customer_cost_df['Activity_Cost'] = (
    customer_cost_df['Quantity'] * customer_cost_df['Cost_per_Unit']
)

customer_cost_df

,Customer Id,Activity Type,Quantity,Cost_per_Unit,Activity_Cost
0,C001,Dispatch,2639.0,2.533336,6685.472777
1,C001,Pick,16790.0,0.300833,5050.988118
2,C001,Receipt,1145.0,2.216662,2538.077511
3,C001,Returns,72.0,6.729235,484.504939
4,C001,Rework,16.0,9.895875,158.334005
...,...,...,...,...,...
205,C030,Receipt,1849.0,2.216662,4098.607265
206,C030,Returns,639.0,6.729235,4299.981338
207,C030,Rework,365.0,9.895875,3611.9945
208,C030,Storage,8051.0,0.031668,254.961615


In [183]:
customer_total_cost = customer_cost_df.groupby('Customer Id').agg(
    Total_Cost_to_Serve=('Activity_Cost', 'sum')
).reset_index()

customer_total_cost

,Customer Id,Total_Cost_to_Serve
0,C001,15635.040752
1,C002,108576.640991
2,C003,58615.833886
3,C004,122745.372106
4,C005,8587.530253
5,C006,27385.672106
6,C007,26317.228469
7,C008,29978.464936
8,C009,30799.796461
9,C010,30708.441566


In [184]:
total_revenue_per_customer

,Customer Id,Total_Revenue
4,C005,441500.81
0,C001,199569.99
1,C002,169240.47
3,C004,154036.11
11,C012,120033.01
12,C013,115901.17
29,C030,114653.57
15,C016,113404.74
21,C022,110145.97
13,C014,106785.08


In [185]:
customer_profit_df = total_revenue_per_customer.merge(
    customer_total_cost,
    on='Customer Id',
    how='left'
)
customer_profit_df

,Customer Id,Total_Revenue,Total_Cost_to_Serve
0,C005,441500.81,8587.530253
1,C001,199569.99,15635.040752
2,C002,169240.47,108576.640991
3,C004,154036.11,122745.372106
4,C012,120033.01,28435.240142
5,C013,115901.17,29500.084748
6,C030,114653.57,31605.195844
7,C016,113404.74,31446.316223
8,C022,110145.97,32853.942049
9,C014,106785.08,28235.533783


In [186]:
m1Inventory_df

,Month,Date,Customer Id,Units On Hand,Avg Days In Stock,Inventory Class
0,Month 1,2026-01-01,C001,4886,30,A
1,Month 1,2026-01-01,C002,2812,5,A
2,Month 1,2026-01-01,C003,2066,53,A
3,Month 1,2026-01-01,C004,4476,25,B
4,Month 1,2026-01-01,C005,3061,84,C
...,...,...,...,...,...,...
925,Month 1,2026-01-31,C026,4876,65,A
926,Month 1,2026-01-31,C027,4935,86,C
927,Month 1,2026-01-31,C028,3864,25,A
928,Month 1,2026-01-31,C029,1930,64,B


In [187]:
# Calculate average inventory metrics per customer
inventory_summary = m1Inventory_df.groupby('Customer Id').agg(
    Avg_Units_On_Hand = ('Units On Hand', 'mean'),
    Avg_Days_In_Stock = ('Avg Days In Stock', 'mean')
).reset_index()

# Create Space Usage metric
inventory_summary['Space_Usage'] = (
    inventory_summary['Avg_Units_On_Hand'] * inventory_summary['Avg_Days_In_Stock']
).round(2)

print(inventory_summary.head(10))

  Customer Id  Avg_Units_On_Hand  Avg_Days_In_Stock  Space_Usage
0        C001        2765.387097          50.129032    138626.18
1        C002        2666.903226          46.709677    124570.19
2        C003        2451.870968          44.451613    108989.62
3        C004        2680.774194          39.903226    106971.54
4        C005        2402.709677          50.129032    120445.51
5        C006        2943.612903          44.516129    131038.25
6        C007        2367.322581          41.516129     98282.07
7        C008        2536.580645          46.838710    118810.16
8        C009        2489.709677          47.548387    118381.68
9        C010        2843.258065          45.967742    130698.15


In [188]:
cost_allocation_df_m1

,Month,Cost Category,Monthly Cost,Cost Type
0,Month 1,Rent,52000,Fixed
1,Month 1,Utilities,12500,Semi-variable
2,Month 1,Equipment,18500,Fixed
3,Month 1,Warehouse Management,34000,Fixed
4,Month 1,Corporate Overhead,28000,Allocated


In [189]:
# Total Space Usage across all customers
total_space_usage = inventory_summary['Space_Usage'].sum()

# Calculate allocation percentage for each customer
inventory_summary['Space_Allocation_%'] = (
    (inventory_summary['Space_Usage'] / total_space_usage) * 100
).round(2)

# Allocate Rent + Equipment ($52,000 + $18,500 = $70,500)
rent_equipment_total = 70500
inventory_summary['Allocated_Space_Cost'] = (
    (inventory_summary['Space_Allocation_%'] / 100) * rent_equipment_total
).round(2)

print(inventory_summary[['Customer Id', 'Space_Usage', 'Space_Allocation_%', 'Allocated_Space_Cost']].head(10))

  Customer Id  Space_Usage  Space_Allocation_%  Allocated_Space_Cost
0        C001    138626.18                3.84               2707.20
1        C002    124570.19                3.45               2432.25
2        C003    108989.62                3.02               2129.10
3        C004    106971.54                2.97               2093.85
4        C005    120445.51                3.34               2354.70
5        C006    131038.25                3.63               2559.15
6        C007     98282.07                2.72               1917.60
7        C008    118810.16                3.29               2319.45
8        C009    118381.68                3.28               2312.40
9        C010    130698.15                3.62               2552.10


In [190]:
# Get total quantity (activity volume) per customer from Activities data
activity_volume = m1Activities_df.groupby('Customer Id')['Quantity'].sum().reset_index()
activity_volume.columns = ['Customer Id', 'Total_Activity_Volume']

# Calculate allocation percentage
total_volume = activity_volume['Total_Activity_Volume'].sum()
activity_volume['Activity_Volume_Allocation_%'] = (
    (activity_volume['Total_Activity_Volume'] / total_volume) * 100
).round(2)

activity_volume

,Customer Id,Total_Activity_Volume,Activity_Volume_Allocation_%
0,C001,40713.0,2.72
1,C002,169509.0,11.31
2,C003,93023.0,6.21
3,C004,158870.0,10.60
4,C005,51971.0,3.47
5,C006,37373.0,2.49
6,C007,35340.0,2.36
7,C008,38882.0,2.59
8,C009,39766.0,2.65
9,C010,39021.0,2.60


In [191]:
# Allocate Warehouse Management cost ($34,000) + utilities (12500)
warehouse_mgmt_total = 46500
activity_volume['Allocated_Warehouse_Mgmt_Cost'] = (
    (activity_volume['Activity_Volume_Allocation_%'] / 100) * warehouse_mgmt_total
).round(2)

print(activity_volume.head(10))

  Customer Id  Total_Activity_Volume  Activity_Volume_Allocation_%  \
0        C001                40713.0                          2.72   
1        C002               169509.0                         11.31   
2        C003                93023.0                          6.21   
3        C004               158870.0                         10.60   
4        C005                51971.0                          3.47   
5        C006                37373.0                          2.49   
6        C007                35340.0                          2.36   
7        C008                38882.0                          2.59   
8        C009                39766.0                          2.65   
9        C010                39021.0                          2.60   

   Allocated_Warehouse_Mgmt_Cost  
0                        1264.80  
1                        5259.15  
2                        2887.65  
3                        4929.00  
4                        1613.55  
5                  

In [192]:
display(customer_profit_df.head()),display(inventory_summary.head()),display(activity_volume.head())

,Customer Id,Total_Revenue,Total_Cost_to_Serve
0,C005,441500.81,8587.530253
1,C001,199569.99,15635.040752
2,C002,169240.47,108576.640991
3,C004,154036.11,122745.372106
4,C012,120033.01,28435.240142


,Customer Id,Avg_Units_On_Hand,Avg_Days_In_Stock,Space_Usage,Space_Allocation_%,Allocated_Space_Cost
0,C001,2765.387097,50.129032,138626.18,3.84,2707.20
1,C002,2666.903226,46.709677,124570.19,3.45,2432.25
2,C003,2451.870968,44.451613,108989.62,3.02,2129.10
3,C004,2680.774194,39.903226,106971.54,2.97,2093.85
4,C005,2402.709677,50.129032,120445.51,3.34,2354.70


,Customer Id,Total_Activity_Volume,Activity_Volume_Allocation_%,Allocated_Warehouse_Mgmt_Cost
0,C001,40713.0,2.72,1264.80
1,C002,169509.0,11.31,5259.15
2,C003,93023.0,6.21,2887.65
3,C004,158870.0,10.60,4929.00
4,C005,51971.0,3.47,1613.55


(None, None, None)

In [193]:
customer_profit_df.isnull().sum()

Customer Id            0
Total_Revenue          0
Total_Cost_to_Serve    0
dtype: int64

In [194]:
inventory_summary.isnull().sum()

Customer Id             0
Avg_Units_On_Hand       0
Avg_Days_In_Stock       0
Space_Usage             0
Space_Allocation_%      0
Allocated_Space_Cost    0
dtype: int64

In [195]:
activity_volume.isnull().sum()

Customer Id                      0
Total_Activity_Volume            0
Activity_Volume_Allocation_%     0
Allocated_Warehouse_Mgmt_Cost    0
dtype: int64

In [196]:
inventory_summary['Customer Id'].nunique()

30

In [197]:
activity_volume['Customer Id'].nunique()

30

In [198]:
# ==========================================
# Merge All Allocated Costs into Main DataFrame
# ==========================================

# main customer profitability data
customer_cost_df = customer_profit_df[['Customer Id', 'Total_Revenue', 'Total_Cost_to_Serve']].copy()

# Merge Allocated Space Cost (Rent + Equipment)
customer_cost_df = customer_cost_df.merge(
    inventory_summary[['Customer Id', 'Allocated_Space_Cost']],
    on='Customer Id',
    how='left'
)

# Merge Allocated Operations Cost (Utilities + Warehouse Management)
customer_cost_df = customer_cost_df.merge(
    activity_volume[['Customer Id', 'Allocated_Warehouse_Mgmt_Cost']],
    on='Customer Id',
    how='left'
)

# Update Total Cost to Serve (add both new allocated costs)
customer_cost_df['Final_Cost_to_Serve'] = (
    customer_cost_df['Total_Cost_to_Serve'] +
    customer_cost_df['Allocated_Space_Cost'] +
    customer_cost_df['Allocated_Warehouse_Mgmt_Cost']
).round(2)

customer_cost_df.head(10)

,Customer Id,Total_Revenue,Total_Cost_to_Serve,Allocated_Space_Cost,Allocated_Warehouse_Mgmt_Cost,Final_Cost_to_Serve
0,C005,441500.81,8587.530253,2354.70,1613.55,12555.78
1,C001,199569.99,15635.040752,2707.20,1264.80,19607.04
2,C002,169240.47,108576.640991,2432.25,5259.15,116268.04
3,C004,154036.11,122745.372106,2093.85,4929.00,129768.22
4,C012,120033.01,28435.240142,2227.80,1306.65,31969.69
5,C013,115901.17,29500.084748,2375.85,1260.15,33136.08
6,C030,114653.57,31605.195844,2657.85,1264.80,35527.85
7,C016,113404.74,31446.316223,2079.75,1264.80,34790.87
8,C022,110145.97,32853.942049,2065.65,1181.10,36100.69
9,C014,106785.08,28235.533783,2305.35,1236.90,31777.78


In [199]:
customer_cost_df['True_Profit']=customer_cost_df['Total_Revenue']-customer_cost_df['Final_Cost_to_Serve']

customer_cost_df['Profit_Margin_%'] = (
    (customer_cost_df['True_Profit'] / customer_cost_df['Total_Revenue']) * 100
).round(2)

customer_cost_df.head()

,Customer Id,Total_Revenue,Total_Cost_to_Serve,Allocated_Space_Cost,Allocated_Warehouse_Mgmt_Cost,Final_Cost_to_Serve,True_Profit,Profit_Margin_%
0,C005,441500.81,8587.530253,2354.70,1613.55,12555.78,428945.03,97.16
1,C001,199569.99,15635.040752,2707.20,1264.80,19607.04,179962.95,90.18
2,C002,169240.47,108576.640991,2432.25,5259.15,116268.04,52972.43,31.3
3,C004,154036.11,122745.372106,2093.85,4929.00,129768.22,24267.89,15.75
4,C012,120033.01,28435.240142,2227.80,1306.65,31969.69,88063.32,73.37


In [200]:
customer_cost_df['Profit_Margin_%'].unique()

array([97.16, 90.18, 31.3, 15.75, 73.37, 71.41, 69.01, 69.32, 67.22,
       70.24, 67.29, 64.91, 67.44, 62.25, 67.84, 65.06, 65.59, 66.0,
       64.98, 65.9, 32.78, 65.31, 62.97, 60.83, 65.73, 62.04, 63.57,
       64.62, 59.7, 65.64], dtype=object)

“We have improved the cost allocation by including labour, space, and operational costs. However, due to data limitations, we could not allocate all overhead costs. As a result, absolute profit margins appear high.
Our focus is on relative comparison between customers rather than absolute margins. Even with partial costs, we can clearly see which customers are significantly better or worse than others.”

Profit-based segmentation logic

In [201]:
def margin_segment(margin):
    if margin < 40:
        return "Problem Customers (<40%)"
    elif 40 <= margin < 67:
        return "Average Customers (40–66%)"
    elif 67 <= margin <= 85:
        return "Strong Customers (67–85%)"
    else:
        return "Premium Customers (>85%)"

customer_cost_df['Margin_Segment'] = customer_cost_df['Profit_Margin_%'].apply(margin_segment)

In [202]:
import plotly.express as px

fig = px.treemap(
    customer_cost_df,
    path=[px.Constant("MONTH 1"), 'Margin_Segment', 'Customer Id'],
    values='Total_Revenue',   # 🔥 FIX: prevents red squeezing
    color='Margin_Segment',

    color_discrete_map={
        "Premium Customers (>85%)": "#00E676",
        "Strong Customers (67–85%)": "#42A5F5",
        "Average Customers (40–66%)": "#FFB300",
        "Problem Customers (<40%)": "#EF5350"
    },

    hover_data={
        "True_Profit": ":,.2f",
        "Profit_Margin_%": ":.2f"
    },

    title="<b>Customer Segmentation – Month 1 Based on Profit_Margin %</b>"
)

# 🔥 CLEAN LOOK (no text inside tiles)
fig.update_traces(
    textinfo="label",   # only customer id + segment, no numbers
    marker=dict(line=dict(width=2, color="black")),

    hovertemplate=
        "<b>Customer: %{label}</b><br><br>" +
        "True Profit: %{customdata[0]:,.2f}<br>" +
        "Profit Margin: %{customdata[1]:.2f}%<extra></extra>"
)

fig.update_layout(
    height=850,
    width=1300,

    paper_bgcolor="black",
    plot_bgcolor="black",

    font=dict(color="white", size=13),

    title=dict(
        x=0.5,
        font=dict(size=22, color="white")
    ),

    margin=dict(t=80, l=10, r=10, b=10)
)

fig.show()

In [203]:
customer_cost_df.head()

,Customer Id,Total_Revenue,Total_Cost_to_Serve,Allocated_Space_Cost,Allocated_Warehouse_Mgmt_Cost,Final_Cost_to_Serve,True_Profit,Profit_Margin_%,Margin_Segment
0,C005,441500.81,8587.530253,2354.70,1613.55,12555.78,428945.03,97.16,Premium Customers (>85%)
1,C001,199569.99,15635.040752,2707.20,1264.80,19607.04,179962.95,90.18,Premium Customers (>85%)
2,C002,169240.47,108576.640991,2432.25,5259.15,116268.04,52972.43,31.3,Problem Customers (<40%)
3,C004,154036.11,122745.372106,2093.85,4929.00,129768.22,24267.89,15.75,Problem Customers (<40%)
4,C012,120033.01,28435.240142,2227.80,1306.65,31969.69,88063.32,73.37,Strong Customers (67–85%)


In [209]:
mean_cost = customer_cost_df['Final_Cost_to_Serve'].mean()
std_cost = customer_cost_df['Final_Cost_to_Serve'].std()

customer_cost_df['Cost_Z'] = (
    customer_cost_df['Final_Cost_to_Serve'] - mean_cost
) / std_cost

def segment(z):
    if z <= -0.5:
        return "Highly Efficient"
    elif z <= 1:
        return "Medium Cost"
    else:
        return "High Cost"
        
customer_cost_df['Cost_Segment'] = customer_cost_df['Cost_Z'].apply(segment)

In [210]:
customer_cost_df['Cost_Segment'].value_counts()

Cost_Segment
Medium Cost         25
High Cost            3
Highly Efficient     2
Name: count, dtype: int64

In [211]:
customer_cost_df.head()

,Customer Id,Total_Revenue,Total_Cost_to_Serve,Allocated_Space_Cost,Allocated_Warehouse_Mgmt_Cost,Final_Cost_to_Serve,True_Profit,Profit_Margin_%,Margin_Segment,Cost_Normalized,Cost_Segment,Cost_Z
0,C005,441500.81,8587.530253,2354.70,1613.55,12555.78,428945.03,97.16,Premium Customers (>85%),0.096755,Highly Efficient,-1.124436
1,C001,199569.99,15635.040752,2707.20,1264.80,19607.04,179962.95,90.18,Premium Customers (>85%),0.151093,Highly Efficient,-0.830317
2,C002,169240.47,108576.640991,2432.25,5259.15,116268.04,52972.43,31.3,Problem Customers (<40%),0.895967,High Cost,3.201567
3,C004,154036.11,122745.372106,2093.85,4929.00,129768.22,24267.89,15.75,Problem Customers (<40%),1.0,High Cost,3.764681
4,C012,120033.01,28435.240142,2227.80,1306.65,31969.69,88063.32,73.37,Strong Customers (67–85%),0.24636,Medium Cost,-0.314651


In [216]:
import plotly.express as px

fig = px.treemap(
    customer_cost_df,
    path=[px.Constant("MONTH 1"), 'Cost_Segment', 'Customer Id'],
    values='Final_Cost_to_Serve',
    color='Cost_Segment',

    color_discrete_map={
        "Highly Efficient": "#43544C",          # green
        "Medium Cost": "#543D07",               # amber
        "High Cost ": "#9D0704"        # red
    },

    title="<b>Customer Cost-to-Serve (Z-Score Segmentation)</b><br><sup>Month 1 Operational Cost Pressure Map</sup>"
)

fig.update_traces(
    textinfo="label",
    marker=dict(line=dict(width=2, color="black")),
    
    hovertemplate=
        "<b>Customer: %{label}</b><br><br>" +
        "Cost to Serve: %{value:,.2f}<extra></extra>"
)

fig.update_layout(
    height=850,
    width=1300,

    paper_bgcolor="black",
    plot_bgcolor="black",

    font=dict(color="white", size=13),

    title=dict(
        x=0.5,
        font=dict(size=22, color="white")
    ),

    margin=dict(t=80, l=10, r=10, b=10)
)

fig.show()

In [217]:
customer_activity_matrix = m1Activities_df.groupby(
    ['Customer Id', 'Activity Type']
)['Quantity'].sum().reset_index()

customer_activity_matrix

,Customer Id,Activity Type,Quantity
0,C001,Dispatch,2639.0
1,C001,Pick,16790.0
2,C001,Receipt,1145.0
3,C001,Returns,72.0
4,C001,Rework,16.0
...,...,...,...
205,C030,Receipt,1849.0
206,C030,Returns,639.0
207,C030,Rework,365.0
208,C030,Storage,8051.0


In [218]:
customer_activity_pivot = customer_activity_matrix.pivot(
    index='Customer Id',
    columns='Activity Type',
    values='Quantity'
).fillna(0)
customer_activity_pivot

Activity Type,Dispatch,Pick,Receipt,Returns,Rework,Storage,Urgent Order
Customer Id,,,,,,,
C001,2639.0,16790.0,1145.0,72.0,16.0,20037.0,14.0
C002,18487.0,132525.0,6389.0,417.0,312.0,11132.0,247.0
C003,8931.0,74247.0,3929.0,310.0,212.0,5294.0,100.0
C004,14876.0,130238.0,2924.0,2664.0,1438.0,5540.0,1190.0
C005,1169.0,8474.0,646.0,22.0,15.0,41640.0,5.0
C006,3029.0,23803.0,1401.0,571.0,334.0,7890.0,345.0
C007,2729.0,23174.0,1545.0,436.0,396.0,6733.0,327.0
C008,3759.0,24937.0,1906.0,439.0,395.0,7170.0,276.0
C009,3666.0,25227.0,2052.0,606.0,331.0,7584.0,300.0


In [219]:
data = {
    'Activity Type': ['Storage', 'Dispatch', 'Receipt', 'Urgent Order', 'Returns', 'Rework', 'Pick'],
    'Category': [
        'Core Value Creator', 
        'Value Creator', 
        'Stable Contributor', 
        'Premium but Costly', 
        'Value Eroder', 
        'Value Destroyer', 
        'Major Value Destroyer'
    ]
}
data

{'Activity Type': ['Storage',
  'Dispatch',
  'Receipt',
  'Urgent Order',
  'Returns',
  'Rework',
  'Pick'],
 'Category': ['Core Value Creator',
  'Value Creator',
  'Stable Contributor',
  'Premium but Costly',
  'Value Eroder',
  'Value Destroyer',
  'Major Value Destroyer']}

In [220]:
activity_map = dict(zip(
    data['Activity Type'],
    data['Category']
))
activity_map

{'Storage': 'Core Value Creator',
 'Dispatch': 'Value Creator',
 'Receipt': 'Stable Contributor',
 'Urgent Order': 'Premium but Costly',
 'Returns': 'Value Eroder',
 'Rework': 'Value Destroyer',
 'Pick': 'Major Value Destroyer'}

In [225]:
customer_category_profile = customer_activity_pivot.copy()

customer_category_profile.columns = customer_category_profile.columns.map(activity_map)

In [226]:
customer_category_profile

Activity Type,Value Creator,Major Value Destroyer,Stable Contributor,Value Eroder,Value Destroyer,Core Value Creator,Premium but Costly
Customer Id,,,,,,,
C001,2639.0,16790.0,1145.0,72.0,16.0,20037.0,14.0
C002,18487.0,132525.0,6389.0,417.0,312.0,11132.0,247.0
C003,8931.0,74247.0,3929.0,310.0,212.0,5294.0,100.0
C004,14876.0,130238.0,2924.0,2664.0,1438.0,5540.0,1190.0
C005,1169.0,8474.0,646.0,22.0,15.0,41640.0,5.0
C006,3029.0,23803.0,1401.0,571.0,334.0,7890.0,345.0
C007,2729.0,23174.0,1545.0,436.0,396.0,6733.0,327.0
C008,3759.0,24937.0,1906.0,439.0,395.0,7170.0,276.0
C009,3666.0,25227.0,2052.0,606.0,331.0,7584.0,300.0


In [227]:
activity_profit

,Activity Type,True_Revenue,Quantity,Labour Cost,Units Processed,Profit,Cost_per_Unit,Revenue per unit
0,Dispatch,524233.02,136774.0,346494.45,136774,177738.57,2.533336,3.832841
1,Pick,142948.00,993588.0,298904.18,993588,-155956.18,0.300833,0.14387
2,Receipt,196552.69,58345.0,129331.12,58345,67221.57,2.216662,3.368801
3,Returns,164834.40,17091.0,115009.36,17091,49825.04,6.729235,9.644515
4,Rework,135953.03,11664.0,115425.49,11664,20527.54,9.895875,11.655781
5,Storage,2293527.44,271771.0,8606.53,271771,2284920.91,0.031668,8.439191
6,Urgent Order,135496.05,9203.0,54643.04,9203,80853.01,5.937525,14.723031


In [229]:
activity_profit_df=activity_profit[['Activity Type','Units Processed','Profit']].copy()

activity_profit_df['Profit_per_Unit'] = (
    activity_profit_df['Profit'] / activity_profit_df['Units Processed']
)
activity_profit_df

,Activity Type,Units Processed,Profit,Profit_per_Unit
0,Dispatch,136774,177738.57,1.299506
1,Pick,993588,-155956.18,-0.156963
2,Receipt,58345,67221.57,1.152139
3,Returns,17091,49825.04,2.915279
4,Rework,11664,20527.54,1.759906
5,Storage,271771,2284920.91,8.407523
6,Urgent Order,9203,80853.01,8.785506


In [230]:
profit_per_unit_map = dict(zip(
    activity_profit_df['Activity Type'],
    activity_profit_df['Profit_per_Unit']
))

profit_per_unit_map

{'Dispatch': 1.2995055346776438,
 'Pick': -0.15696262434731498,
 'Receipt': 1.1521393435598593,
 'Returns': 2.915279386811771,
 'Rework': 1.75990569272977,
 'Storage': 8.407522914512587,
 'Urgent Order': 8.785505813321741}

In [234]:
customer_profit_impact = customer_activity_pivot.copy()

for activity in customer_profit_impact.columns:
    customer_profit_impact[activity] = (
        customer_profit_impact[activity] * profit_per_unit_map[activity]
    )
customer_profit_impact['Net_Activity_Profit_Impact'] = customer_profit_impact.sum(axis=1)

In [235]:
customer_profit_impact

Activity Type,Dispatch,Pick,Receipt,Returns,Rework,Storage,Urgent Order,Net_Activity_Profit_Impact
Customer Id,,,,,,,,
C001,3429.395106,-2635.402463,1319.199548,209.900116,28.158491,168461.536638,122.997081,170935.784518
C002,24023.958820,-20801.471792,7361.018266,1215.671504,549.090576,93592.545084,2170.019936,108110.832395
C003,11605.883930,-11654.003970,4526.755481,903.736610,373.100007,44509.426309,878.550581,51143.448949
C004,19331.444334,-20442.498270,3368.855441,7766.304286,2530.744386,46577.676946,10454.751918,69587.279042
C005,1519.121970,-1330.101279,744.282016,64.136147,26.398585,350089.254160,43.927529,351157.019129
C006,3936.202265,-3736.181347,1614.147220,1664.624530,587.808501,66335.355796,3030.999506,73432.956470
C007,3546.350604,-3637.451857,1780.055286,1271.061813,696.922654,56607.851783,2872.860401,63137.650685
C008,4884.841305,-3914.176963,2195.977589,1279.807651,695.162749,60281.939297,2424.799604,67848.351231
C009,4763.987290,-3959.696124,2364.189933,1766.659308,582.528784,63762.653784,2635.651744,71915.974719


In [236]:
driver_summary = customer_profit_impact.drop(columns=['Net_Activity_Profit_Impact']).sum().sort_values()
driver_summary

Activity Type
Pick            -155956.18
Rework            20527.54
Returns           49825.04
Receipt           67221.57
Urgent Order      80853.01
Dispatch         177738.57
Storage         2284920.91
dtype: float64

In [239]:
df_long = customer_profit_impact.drop(
    columns=['Net_Activity_Profit_Impact']
).reset_index().melt(
    id_vars='Customer Id',
    var_name='Activity',
    value_name='Profit Impact'
)
df_long

,Customer Id,Activity,Profit Impact
0,C001,Dispatch,3429.395106
1,C002,Dispatch,24023.958820
2,C003,Dispatch,11605.883930
3,C004,Dispatch,19331.444334
4,C005,Dispatch,1519.121970
...,...,...,...
205,C026,Urgent Order,2521.440168
206,C027,Urgent Order,2969.500965
207,C028,Urgent Order,3312.135692
208,C029,Urgent Order,2943.144447


In [249]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# prepare data (wide format)
df = customer_profit_impact.drop(columns=['Net_Activity_Profit_Impact'])

customers = df.index.tolist()
activities = df.columns.tolist()

# 6 rows × 5 cols = 30 customers
fig = make_subplots(
    rows=6,
    cols=5,
    subplot_titles=customers
)

row = 1
col = 1

for i, cust in enumerate(customers):
    fig.add_trace(
        go.Bar(
            x=activities,
            y=df.loc[cust],
            marker_color=df.loc[cust].apply(
                lambda x: "green" if x > 0 else "red"
            )
        ),
        row=row,
        col=col
    )

    col += 1
    if col > 5:
        col = 1
        row += 1

fig.update_layout(
    title="Customer Profit Impact by Activity (All Customers)",
    paper_bgcolor="black",
    plot_bgcolor="black",
    font=dict(color="white"),
    height=1400,
    width=2000,
    showlegend=False
)

fig.show()